<a href="https://colab.research.google.com/github/muzazr/favera-model/blob/main/04_candidate_v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 Candidate V4: Global Transaction Risk Model

Eksperimen ini menggunakan fitur Candidate V3 sebagai basis.

Perubahan utama Candidate V4:

1. Feature diagnostic.
2. Deployable interaction features.
3. Controlled LightGBM benchmark.
4. Refit menggunakan train dan validation.
5. Threshold dipilih hanya dari calibration.
6. Test digunakan sekali untuk evaluasi final.

Dataset: MoMTSim V1  
Scope model: TRANSFER  
Label: Synthetic fraud, bukan transaksi judi.

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q \
    polars \
    pyarrow \
    pandas \
    numpy \
    scikit-learn \
    lightgbm \
    shap \
    matplotlib \
    joblib

In [3]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/favera/model/transaction-anomaly"
)

RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "models"
OUTPUT_DIR = PROJECT_DIR / "outputs"

for directory in [
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    MODEL_DIR,
    OUTPUT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

In [4]:
import random
import joblib
import numpy as np
import pandas as pd
import polars as pl

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score,
    precision_score,
    recall_score,
)
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

In [5]:
DATA_PATH = RAW_DATA_DIR / "momtsim_v1.csv"

lazy_df = pl.scan_csv(
    DATA_PATH,
    infer_schema_length=10_000,
    ignore_errors=True
)

print(lazy_df.collect_schema())

sample_df = lazy_df.head(5).collect()
sample_df

row_count = lazy_df.select(pl.len()).collect().item()
print(f"jumlah transaksi: {row_count:,}")

Schema({'step': Int64, 'transactionType': String, 'amount': Float64, 'initiator': Int64, 'oldBalInitiator': Float64, 'newBalInitiator': Float64, 'recipient': String, 'oldBalRecipient': Float64, 'newBalRecipient': Float64, 'isFraud': Int64})
jumlah transaksi: 1,720,181


## 2. Memahami Struktur Dataset

In [6]:
from IPython.display import display, Markdown
import polars as pl

schema = lazy_df.collect_schema()
columns = schema.names()

schema_df = pl.DataFrame({
    "column": columns,
    "data_type": [str(schema[col]) for col in columns]
})

display(Markdown(
    f"Dataset memiliki **{row_count:,} baris** dan "
    f"**{len(columns)} kolom**."
))

display(schema_df)

Dataset memiliki **1,720,181 baris** dan **10 kolom**.

column,data_type
str,str
"""step""","""Int64"""
"""transactionType""","""String"""
"""amount""","""Float64"""
"""initiator""","""Int64"""
"""oldBalInitiator""","""Float64"""
"""newBalInitiator""","""Float64"""
"""recipient""","""String"""
"""oldBalRecipient""","""Float64"""
"""newBalRecipient""","""Float64"""


In [7]:
preview_df = lazy_df.head(10).collect()

display(preview_df)

EDA_SAMPLE_SIZE = 100_000

eda_df = lazy_df.head(EDA_SAMPLE_SIZE).collect(
    engine="streaming"
)

display(Markdown(
    f"Sampel EDA menggunakan **{eda_df.height:,} baris**."
))

step,transactionType,amount,initiator,oldBalInitiator,newBalInitiator,recipient,oldBalRecipient,newBalRecipient,isFraud
i64,str,f64,i64,f64,f64,str,f64,f64,i64
0,"""TRANSFER""",19824.96,4537027967639631,187712.18,167887.22,"""4875702729424478""",8.31,19833.27,1
0,"""PAYMENT""",598.97,4296267625767470,8.92,8.92,"""25-0000401""",0.0,0.0,0
0,"""PAYMENT""",545.85,4178224023847746,93.6,-452.25,"""13-0001587""",0.0,545.85,0
0,"""TRANSFER""",19847.01,4178224023847746,-452.25,-20299.26,"""4096920916696293""",4011.72,23858.74,1
0,"""PAYMENT""",546.89,4779013371563747,159148.76,158601.88,"""75-0003564""",0.0,546.89,0
0,"""TRANSFER""",16973.75,4779013371563747,158601.88,141628.12,"""4593247015015219""",54.33,17028.08,1
0,"""PAYMENT""",659.02,4479448076398077,261950.48,261291.46,"""11-0003866""",0.0,659.02,0
0,"""TRANSFER""",21103.99,4479448076398077,261291.46,240187.47,"""4649550475321678""",19.63,21123.63,1
0,"""PAYMENT""",648.89,4587942659105635,4808.94,4160.05,"""64-0008405""",0.0,648.89,0


Sampel EDA menggunakan **100,000 baris**.

In [8]:
null_summary = (
    lazy_df
    .select([
        pl.col(col).null_count().alias(col)
        for col in columns
    ])
    .collect(engine="streaming")
    .transpose(
        include_header=True,
        header_name="column",
        column_names=["null_count"]
    )
    .with_columns(
        (
            pl.col("null_count") / row_count * 100
        ).round(4).alias("null_percentage")
    )
    .sort("null_count", descending=True)
)

columns_with_null = null_summary.filter(
    pl.col("null_count") > 0
)

if columns_with_null.height == 0:
    display(Markdown("**Tidak ditemukan missing value.**"))
else:
    display(columns_with_null)

**Tidak ditemukan missing value.**

In [9]:
sample_schema = eda_df.schema

numeric_cols = [
    col
    for col, dtype in sample_schema.items()
    if dtype.is_numeric()
]

string_cols = [
    col
    for col, dtype in sample_schema.items()
    if dtype == pl.String
]

print("Kolom numerik:", numeric_cols)
print("Kolom teks:", string_cols)



Kolom numerik: ['step', 'amount', 'initiator', 'oldBalInitiator', 'newBalInitiator', 'oldBalRecipient', 'newBalRecipient', 'isFraud']
Kolom teks: ['transactionType', 'recipient']


In [10]:
if numeric_cols:
    numeric_summary = eda_df.select(
        numeric_cols
    ).describe()

    display(numeric_summary)
else:
    display(Markdown(
        "**Tidak ditemukan kolom numerik pada hasil parsing awal.**"
    ))

statistic,step,amount,initiator,oldBalInitiator,newBalInitiator,oldBalRecipient,newBalRecipient,isFraud
str,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",5.56527,56655.657033,4.5010e15,287459.142625,332396.509199,7835.952872,12129.0505,0.11091
"""std""",2.983908,80062.006032,2.9027e14,569433.351835,601407.309418,37099.060757,42482.275094,0.314022
"""min""",0.0,1.36,4.0001e15,-199997.11,-199997.11,-149882.02,-135727.98,0.0
"""25%""",3.0,598.6,4.2472e15,18.44,20.4,534.48,1159.69,0.0
"""50%""",7.0,17572.83,4.5055e15,37481.59,83645.61,1735.43,2377.61,0.0
"""75%""",8.0,91043.64,4.7535e15,340451.41,425667.37,3303.1,4703.7,0.0
"""max""",9.0,1308624.7,4.9999e15,5.5312e6,5662492.3,1.5432e6,2.6180e6,1.0


In [11]:
if string_cols:
    unique_summary = pl.DataFrame({
        "column": string_cols,
        "unique_count_sample": [
            eda_df[col].n_unique()
            for col in string_cols
        ]
    }).sort(
        "unique_count_sample",
        descending=True
    )

    display(unique_summary)

column,unique_count_sample
str,i64
"""recipient""",10090
"""transactionType""",5


In [12]:
duplicate_count_sample = int(
    eda_df.is_duplicated().sum()
)

duplicate_percentage = (
    duplicate_count_sample / eda_df.height * 100
)

display(Markdown(
    f"""
**Duplikasi identik pada sampel:**
- Jumlah: **{duplicate_count_sample:,} baris**
- Persentase: **{duplicate_percentage:.4f}%**
"""
))


**Duplikasi identik pada sampel:**
- Jumlah: **0 baris**
- Persentase: **0.0000%**


In [13]:
target_keywords = [
    "fraud",
    "label",
    "class",
    "target",
    "anomaly"
]

time_keywords = [
    "date",
    "time",
    "timestamp",
    "step"
]

target_candidates = [
    col for col in columns
    if any(keyword in col.lower() for keyword in target_keywords)
]

time_candidates = [
    col for col in columns
    if any(keyword in col.lower() for keyword in time_keywords)
]

display(Markdown(
    f"""
### Kandidat Kolom Penting

**Kandidat label:** `{target_candidates}`

**Kandidat waktu:** `{time_candidates}`
"""
))


### Kandidat Kolom Penting

**Kandidat label:** `['isFraud']`

**Kandidat waktu:** `['step']`


In [14]:
for target_col in target_candidates:
    target_distribution = (
        lazy_df
        .group_by(target_col)
        .agg(
            pl.len().alias("count")
        )
        .with_columns(
            (
                pl.col("count") / row_count * 100
            ).round(4).alias("percentage")
        )
        .sort("count", descending=True)
        .collect(engine="streaming")
    )

    display(Markdown(
        f"### Distribusi `{target_col}`"
    ))
    display(target_distribution)

### Distribusi `isFraud`

isFraud,count,percentage
i64,u32,f64
0,1544663,89.7965
1,175518,10.2035


In [15]:
import re


def normalize_column_name(name: str) -> str:
    """Mengubah nama kolom menjadi format snake_case."""
    name = name.strip()
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", name)
    name = re.sub(r"[^a-zA-Z0-9]+", "_", name)
    return name.strip("_").lower()


rename_map = {
    col: normalize_column_name(col)
    for col in columns
}

clean_lf = lazy_df.rename(rename_map)

display(
    pl.DataFrame({
        "original_column": list(rename_map.keys()),
        "clean_column": list(rename_map.values())
    })
)

original_column,clean_column
str,str
"""step""","""step"""
"""transactionType""","""transaction_type"""
"""amount""","""amount"""
"""initiator""","""initiator"""
"""oldBalInitiator""","""old_bal_initiator"""
"""newBalInitiator""","""new_bal_initiator"""
"""recipient""","""recipient"""
"""oldBalRecipient""","""old_bal_recipient"""
"""newBalRecipient""","""new_bal_recipient"""


In [16]:
clean_schema = clean_lf.collect_schema()

clean_string_cols = [
    col
    for col, dtype in clean_schema.items()
    if dtype == pl.String
]

NULL_TOKENS = [
    "",
    "na",
    "n/a",
    "null",
    "none",
    "-"
]

clean_lf = clean_lf.with_columns([
    pl.when(
        pl.col(col)
        .str.strip_chars()
        .str.to_lowercase()
        .is_in(NULL_TOKENS)
    )
    .then(None)
    .otherwise(
        pl.col(col).str.strip_chars()
    )
    .alias(col)
    for col in clean_string_cols
])

display(Markdown(
    f"Kolom teks yang dibersihkan: **{len(clean_string_cols)} kolom**."
))

Kolom teks yang dibersihkan: **2 kolom**.

In [17]:
STAGE_1_PATH = (
    PROCESSED_DATA_DIR /
    "momtsim_stage1_clean.parquet"
)

clean_lf.sink_parquet(
    str(STAGE_1_PATH),
    compression="zstd"
)

display(Markdown(
    f"""
### Data Cleaning Tahap Awal Selesai

File tersimpan di:

`{STAGE_1_PATH}`
"""
))

clean_lf = pl.scan_parquet(
    STAGE_1_PATH
)


### Data Cleaning Tahap Awal Selesai

File tersimpan di:

`/content/drive/MyDrive/favera/model/transaction-anomaly/data/processed/momtsim_stage1_clean.parquet`


In [18]:
import polars as pl
from IPython.display import display, Markdown

# Kolom utama berdasarkan dataset kamu
TIME_COL = "step"
TARGET_COL = "is_fraud"

# Pastikan kolom wajib tersedia
required_columns = [
    "step",
    "transaction_type",
    "amount",
    "initiator",
    "old_bal_initiator",
    "new_bal_initiator",
    "recipient",
    "old_bal_recipient",
    "new_bal_recipient",
    "is_fraud",
]

available_columns = clean_lf.collect_schema().names()

missing_columns = [
    col for col in required_columns
    if col not in available_columns
]

if missing_columns:
    raise ValueError(
        f"Kolom tidak ditemukan: {missing_columns}"
    )

print("Semua kolom wajib tersedia.")

Semua kolom wajib tersedia.


In [19]:
# ============================================================
# FEATURE ENGINEERING AWAL
# ============================================================

feature_lf = (
    clean_lf
    .with_columns([
        # Tipe data dibuat konsisten
        pl.col("step").cast(pl.Int64),
        pl.col("amount").cast(pl.Float64),
        pl.col("initiator").cast(pl.String),
        pl.col("recipient").cast(pl.String),
        pl.col("is_fraud").cast(pl.Int8).alias("target"),

        pl.col("transaction_type")
        .cast(pl.String)
        .str.strip_chars()
        .str.to_uppercase(),
    ])
    .filter(
        pl.col("step").is_not_null() &
        pl.col("amount").is_not_null() &
        pl.col("target").is_not_null()
    )
    .with_columns([
        # Mengurangi efek nominal yang sangat ekstrem
        pl.col("amount")
        .clip(lower_bound=0)
        .log1p()
        .alias("amount_log1p"),

        # Perubahan saldo aktual
        (
            pl.col("new_bal_initiator") -
            pl.col("old_bal_initiator")
        ).alias("initiator_balance_change"),

        (
            pl.col("new_bal_recipient") -
            pl.col("old_bal_recipient")
        ).alias("recipient_balance_change"),

        # Rasio nominal terhadap saldo awal
        (
            pl.col("amount") /
            (pl.col("old_bal_initiator").abs() + 1)
        ).alias("amount_to_initiator_balance_ratio"),

        (
            pl.col("amount") /
            (pl.col("old_bal_recipient").abs() + 1)
        ).alias("amount_to_recipient_balance_ratio"),

        # Seberapa besar saldo pengirim berkurang
        (
            (
                pl.col("old_bal_initiator") -
                pl.col("new_bal_initiator")
            )
            / (pl.col("old_bal_initiator").abs() + 1)
        ).alias("initiator_balance_drop_ratio"),

        # Penanda kondisi saldo tertentu
        (
            pl.col("old_bal_initiator") <= 0
        ).cast(pl.Int8).alias("initiator_old_balance_non_positive"),

        (
            pl.col("new_bal_initiator") < 0
        ).cast(pl.Int8).alias("initiator_new_balance_negative"),

        (
            pl.col("old_bal_recipient") == 0
        ).cast(pl.Int8).alias("recipient_old_balance_zero"),
    ])
)

In [20]:
feature_lf = feature_lf.with_columns([
    # Saldo pengirim yang secara sederhana diharapkan
    (
        pl.col("old_bal_initiator") -
        pl.col("amount")
    ).alias("expected_new_bal_initiator"),

    # Saldo penerima yang secara sederhana diharapkan
    (
        pl.col("old_bal_recipient") +
        pl.col("amount")
    ).alias("expected_new_bal_recipient"),
])

feature_lf = feature_lf.with_columns([
    # Selisih antara saldo aktual dan saldo yang diharapkan
    (
        pl.col("new_bal_initiator") -
        pl.col("expected_new_bal_initiator")
    ).abs().alias("initiator_balance_error"),

    (
        pl.col("new_bal_recipient") -
        pl.col("expected_new_bal_recipient")
    ).abs().alias("recipient_balance_error"),
])

In [21]:
import numpy as np
import polars as pl

# ============================================================
# FITUR WAKTU BERDASARKAN STEP
# 1 step MoMTSim = 1 jam
# ============================================================

feature_lf = feature_lf.with_columns([
    # Jam relatif dalam siklus 24 jam.
    # Ini belum tentu menunjukkan jam kalender sebenarnya
    # karena titik awal simulasi belum dikonfirmasi.
    (
        pl.col("step") % 24
    )
    .cast(pl.Int8)
    .alias("hour_of_day"),

    # Hari relatif sejak simulasi dimulai.
    (
        pl.col("step") // 24
    )
    .cast(pl.Int16)
    .alias("day_index"),
])

feature_lf = feature_lf.with_columns([
    # Fitur eksperimental.
    # Tidak akan dimasukkan ke global model V3 dahulu.
    (
        (pl.col("hour_of_day") >= 22)
        |
        (pl.col("hour_of_day") < 5)
    )
    .cast(pl.Int8)
    .alias("is_night"),

    # Representasi siklik.
    # Jam 23 dibuat dekat dengan jam 0.
    (
        2
        * np.pi
        * pl.col("hour_of_day")
        / 24
    )
    .sin()
    .alias("hour_sin"),

    (
        2
        * np.pi
        * pl.col("hour_of_day")
        / 24
    )
    .cos()
    .alias("hour_cos"),
])

# ============================================================
# PEMERIKSAAN STEP DATASET
# ============================================================

step_summary = (
    feature_lf
    .select([
        pl.col("step")
        .min()
        .alias("minimum_step"),

        pl.col("step")
        .max()
        .alias("maximum_step"),

        pl.col("step")
        .n_unique()
        .alias("unique_steps"),

        pl.col("hour_of_day")
        .min()
        .alias("minimum_relative_hour"),

        pl.col("hour_of_day")
        .max()
        .alias("maximum_relative_hour"),

        pl.col("day_index")
        .max()
        .alias("maximum_day_index"),
    ])
    .collect(
        engine="streaming"
    )
)

display(step_summary)

minimum_step,maximum_step,unique_steps,minimum_relative_hour,maximum_relative_hour,maximum_day_index
i64,i64,u32,i8,i8,i16
0,143,144,0,23,5


In [22]:
# ============================================================
# 00.1 CONFIGURATION AND EXPERIMENT SETUP V4
# ============================================================

import gc
import json
import random
import joblib
import numpy as np
import pandas as pd
import polars as pl

from IPython.display import display

from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
)

from lightgbm import (
    LGBMClassifier,
    early_stopping,
    log_evaluation,
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

EXPERIMENT_VERSION = "candidate_v4_transfer_global"
DATASET_VERSION = "momtsim_v1"
BASE_FEATURE_VERSION = "candidate_v3"

# ============================================================
# TARGET SEMENTARA GLOBAL SUPERVISED MODEL
# ============================================================

TARGET_PR_AUC = 0.50
TARGET_PRECISION = 0.50
TARGET_RECALL = 0.70
TARGET_F1 = 0.58
TARGET_FPR = 0.20

# ============================================================
# TARGET FINAL SETELAH SCORE FUSION
# ============================================================

APPLICATION_TARGETS = {
    "pr_auc": 0.60,
    "precision": 0.65,
    "recall": 0.80,
    "f1": 0.70,
    "false_positive_rate": 0.10,
    "false_alerts_per_user_week": 2.0,
}

# ============================================================
# BASELINE TEST CANDIDATE V3
# Disalin dari hasil final V3.
# Tidak dihitung ulang dalam eksperimen V4.
# ============================================================

V3_BASELINE_TEST_METRICS = {
    "pr_auc": 0.348577,
    "precision": 0.307330,
    "recall": 0.998952,
    "f1": 0.470048,
    "false_positive_rate": 0.897196,
    "alert_rate": 0.926191,
}

print("Experiment:", EXPERIMENT_VERSION)
print("Dataset:", DATASET_VERSION)
print("Base features:", BASE_FEATURE_VERSION)
print("Seed:", SEED)

Experiment: candidate_v4_transfer_global
Dataset: momtsim_v1
Base features: candidate_v3
Seed: 42


In [23]:
# ============================================================
# 00.2 OPERATING POINT TABLE UTILITY
# ============================================================
def build_operating_point_table(
    y_true,
    probabilities,
):
    """
    Membuat seluruh operating point berdasarkan kemungkinan threshold.
    """
    y_true = np.asarray(
        y_true,
        dtype=np.int8,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    if len(y_true) != len(probabilities):
        raise ValueError(
            "Jumlah target dan probabilitas tidak sama."
        )

    # Skor tertinggi diletakkan lebih dahulu.
    order = np.argsort(
        -probabilities,
        kind="mergesort",
    )

    sorted_target = y_true[order]
    sorted_probability = probabilities[order]

    true_positive = np.cumsum(
        sorted_target
    )

    false_positive = np.cumsum(
        1 - sorted_target
    )

    total_positive = int(
        sorted_target.sum()
    )

    total_negative = (
        len(sorted_target)
        - total_positive
    )

    false_negative = (
        total_positive
        - true_positive
    )

    true_negative = (
        total_negative
        - false_positive
    )

    precision = (
        true_positive /
        np.maximum(
            true_positive + false_positive,
            1,
        )
    )

    recall = (
        true_positive /
        max(total_positive, 1)
    )

    false_positive_rate = (
        false_positive /
        max(total_negative, 1)
    )

    f1 = (
        2 * precision * recall /
        np.maximum(
            precision + recall,
            1e-12,
        )
    )

    # Ambil posisi terakhir untuk probabilitas yang sama.
    distinct_indices = np.r_[
        np.flatnonzero(
            sorted_probability[:-1]
            != sorted_probability[1:]
        ),
        len(sorted_probability) - 1,
    ]

    return pd.DataFrame({
        "threshold": (
            sorted_probability[
                distinct_indices
            ]
        ),
        "precision": (
            precision[
                distinct_indices
            ]
        ),
        "recall": (
            recall[
                distinct_indices
            ]
        ),
        "f1": (
            f1[
                distinct_indices
            ]
        ),
        "false_positive_rate": (
            false_positive_rate[
                distinct_indices
            ]
        ),
        "true_positive": (
            true_positive[
                distinct_indices
            ]
        ),
        "false_positive": (
            false_positive[
                distinct_indices
            ]
        ),
        "false_negative": (
            false_negative[
                distinct_indices
            ]
        ),
        "true_negative": (
            true_negative[
                distinct_indices
            ]
        ),
    })

# Histori Pengirim

In [24]:
required_history_columns = [
    "step",
    "initiator",
    "recipient",
    "amount",
]

available_feature_columns = (
    feature_lf
    .collect_schema()
    .names()
)

missing_history_columns = [
    column
    for column in required_history_columns
    if column not in available_feature_columns
]

if missing_history_columns:
    raise ValueError(
        "Kolom histori tidak ditemukan: "
        f"{missing_history_columns}"
    )

# Materialisasi hanya empat kolom yang diperlukan.
# Ini mencegah seluruh feature_lf masuk RAM sekaligus.
base_history_df = (
    feature_lf
    .select(
        required_history_columns
    )
    .collect(
        engine="streaming"
    )
)

print(
    "History source shape:",
    base_history_df.shape,
)

initiator_step_df = (
    base_history_df
    .group_by([
        "initiator",
        "step",
    ])
    .agg([
        pl.len()
        .alias(
            "initiator_tx_count_step"
        ),

        pl.col("amount")
        .sum()
        .alias(
            "initiator_amount_sum_step"
        ),

        pl.col("amount")
        .pow(2)
        .sum()
        .alias(
            "initiator_amount_sq_sum_step"
        ),

        pl.col("amount")
        .max()
        .alias(
            "initiator_amount_max_step"
        ),
    ])
    .sort([
        "initiator",
        "step",
    ])
)

print(
    "Initiator-step rows:",
    initiator_step_df.height,
)

History source shape: (1720181, 4)
Initiator-step rows: 558223


In [25]:
initiator_history_df = (
    initiator_step_df
    .with_columns([
        (
            pl.col(
                "initiator_tx_count_step"
            )
            .cum_sum()
            .over(
                "initiator",
                order_by="step",
            )
            -
            pl.col(
                "initiator_tx_count_step"
            )
        )
        .alias(
            "initiator_tx_count_prior"
        ),

        (
            pl.col(
                "initiator_amount_sum_step"
            )
            .cum_sum()
            .over(
                "initiator",
                order_by="step",
            )
            -
            pl.col(
                "initiator_amount_sum_step"
            )
        )
        .alias(
            "initiator_amount_sum_prior"
        ),

        (
            pl.col(
                "initiator_amount_sq_sum_step"
            )
            .cum_sum()
            .over(
                "initiator",
                order_by="step",
            )
            -
            pl.col(
                "initiator_amount_sq_sum_step"
            )
        )
        .alias(
            "initiator_amount_sq_sum_prior"
        ),

        pl.col(
            "initiator_amount_max_step"
        )
        .cum_max()
        .shift(1)
        .over(
            "initiator",
            order_by="step",
        )
        .alias(
            "initiator_amount_max_prior"
        ),

        pl.col("step")
        .shift(1)
        .over(
            "initiator",
            order_by="step",
        )
        .alias(
            "initiator_previous_step"
        ),
    ])
)

In [26]:
initiator_6h_df = (
    initiator_step_df
    .select([
        "initiator",
        "step",
        "initiator_tx_count_step",
        "initiator_amount_sum_step",
    ])
    .sort([
        "initiator",
        "step",
    ])
    .rolling(
        index_column="step",
        period="6i",
        closed="left",
        group_by="initiator",
    )
    .agg([
        pl.col(
            "initiator_tx_count_step"
        )
        .sum()
        .alias(
            "initiator_tx_count_6h"
        ),

        pl.col(
            "initiator_amount_sum_step"
        )
        .sum()
        .alias(
            "initiator_amount_sum_6h"
        ),
    ])
)

In [27]:
initiator_24h_df = (
    initiator_step_df
    .select([
        "initiator",
        "step",
        "initiator_tx_count_step",
        "initiator_amount_sum_step",
    ])
    .sort([
        "initiator",
        "step",
    ])
    .rolling(
        index_column="step",
        period="24i",
        closed="left",
        group_by="initiator",
    )
    .agg([
        pl.col(
            "initiator_tx_count_step"
        )
        .sum()
        .alias(
            "initiator_tx_count_24h"
        ),

        pl.col(
            "initiator_amount_sum_step"
        )
        .sum()
        .alias(
            "initiator_amount_sum_24h"
        ),
    ])
)

In [28]:
initiator_history_df = (
    initiator_history_df
    .join(
        initiator_6h_df,
        on=[
            "initiator",
            "step",
        ],
        how="left",
    )
    .join(
        initiator_24h_df,
        on=[
            "initiator",
            "step",
        ],
        how="left",
    )
)

# Histori Penerima

In [29]:
recipient_step_df = (
    base_history_df
    .group_by([
        "recipient",
        "step",
    ])
    .agg([
        pl.len()
        .alias(
            "recipient_tx_count_step"
        ),
    ])
    .sort([
        "recipient",
        "step",
    ])
)

In [30]:
recipient_history_df = (
    recipient_step_df
    .with_columns([
        (
            pl.col(
                "recipient_tx_count_step"
            )
            .cum_sum()
            .over(
                "recipient",
                order_by="step",
            )
            -
            pl.col(
                "recipient_tx_count_step"
            )
        )
        .alias(
            "recipient_tx_count_prior"
        ),
    ])
    .select([
        "recipient",
        "step",
        "recipient_tx_count_prior",
    ])
)

# Histori Pasangan pengirim dan Penerima

In [31]:
pair_step_df = (
    base_history_df
    .group_by([
        "initiator",
        "recipient",
        "step",
    ])
    .agg([
        pl.len()
        .alias(
            "pair_tx_count_step"
        ),
    ])
    .sort([
        "initiator",
        "recipient",
        "step",
    ])
)

pair_history_df = (
    pair_step_df
    .with_columns([
        (
            pl.col(
                "pair_tx_count_step"
            )
            .cum_sum()
            .over(
                [
                    "initiator",
                    "recipient",
                ],
                order_by="step",
            )
            -
            pl.col(
                "pair_tx_count_step"
            )
        )
        .alias(
            "pair_tx_count_prior"
        ),

        pl.col("step")
        .shift(1)
        .over(
            [
                "initiator",
                "recipient",
            ],
            order_by="step",
        )
        .alias(
            "pair_previous_step"
        ),
    ])
    .select([
        "initiator",
        "recipient",
        "step",
        "pair_tx_count_prior",
        "pair_previous_step",
    ])
)

In [32]:
behavioral_feature_lf = (
    feature_lf
    .join(
        initiator_history_df.lazy(),
        on=[
            "initiator",
            "step",
        ],
        how="left",
    )
    .join(
        recipient_history_df.lazy(),
        on=[
            "recipient",
            "step",
        ],
        how="left",
    )
    .join(
        pair_history_df.lazy(),
        on=[
            "initiator",
            "recipient",
            "step",
        ],
        how="left",
    )
)

# Pengisian Null Value

In [33]:
history_zero_columns = [
    "initiator_tx_count_prior",
    "initiator_amount_sum_prior",
    "initiator_amount_sq_sum_prior",
    "initiator_amount_max_prior",
    "initiator_tx_count_6h",
    "initiator_amount_sum_6h",
    "initiator_tx_count_24h",
    "initiator_amount_sum_24h",
    "recipient_tx_count_prior",
    "pair_tx_count_prior",
]

behavioral_feature_lf = (
    behavioral_feature_lf
    .with_columns([
        pl.col(column)
        .fill_null(0)
        .alias(column)
        for column in history_zero_columns
    ])
)

# Fitur Behavioral V2

In [34]:
behavioral_feature_lf = (
    behavioral_feature_lf
    .with_columns([
        (
            pl.col(
                "initiator_tx_count_prior"
            ) > 0
        )
        .cast(pl.Int8)
        .alias(
            "has_initiator_history"
        ),

        (
            pl.col(
                "pair_tx_count_prior"
            ) == 0
        )
        .cast(pl.Int8)
        .alias(
            "is_new_recipient"
        ),

        pl.when(
            pl.col(
                "initiator_tx_count_prior"
            ) > 0
        )
        .then(
            pl.col(
                "initiator_amount_sum_prior"
            )
            /
            pl.col(
                "initiator_tx_count_prior"
            )
        )
        .otherwise(0)
        .alias(
            "initiator_amount_mean_prior"
        ),
    ])
)

behavioral_feature_lf = (
    behavioral_feature_lf
    .with_columns([
        pl.when(
            pl.col(
                "initiator_tx_count_prior"
            ) > 1
        )
        .then(
            (
                pl.col(
                    "initiator_amount_sq_sum_prior"
                )
                /
                pl.col(
                    "initiator_tx_count_prior"
                )
            )
            -
            pl.col(
                "initiator_amount_mean_prior"
            ).pow(2)
        )
        .otherwise(0)
        .clip(lower_bound=0)
        .alias(
            "initiator_amount_variance_prior"
        ),
    ])
    .with_columns([
        pl.col(
            "initiator_amount_variance_prior"
        )
        .sqrt()
        .alias(
            "initiator_amount_std_prior"
        ),
    ])
)

behavioral_feature_lf = (
    behavioral_feature_lf
    .with_columns([
        pl.when(
            pl.col(
                "initiator_tx_count_prior"
            ) > 0
        )
        .then(
            pl.col("amount")
            /
            (
                pl.col(
                    "initiator_amount_mean_prior"
                ).abs()
                + 1
            )
        )
        .otherwise(0)
        .alias(
            "amount_vs_initiator_mean"
        ),

        pl.when(
            pl.col(
                "initiator_tx_count_prior"
            ) > 1
        )
        .then(
            (
                pl.col("amount")
                -
                pl.col(
                    "initiator_amount_mean_prior"
                )
            )
            /
            (
                pl.col(
                    "initiator_amount_std_prior"
                )
                + 1
            )
        )
        .otherwise(0)
        .alias(
            "amount_zscore_initiator"
        ),

        pl.when(
            pl.col(
                "initiator_amount_max_prior"
            ) > 0
        )
        .then(
            pl.col("amount")
            /
            (
                pl.col(
                    "initiator_amount_max_prior"
                )
                + 1
            )
        )
        .otherwise(0)
        .alias(
            "amount_vs_initiator_max"
        ),

        pl.when(
            pl.col(
                "initiator_previous_step"
            ).is_not_null()
        )
        .then(
            pl.col("step")
            -
            pl.col(
                "initiator_previous_step"
            )
        )
        .otherwise(-1)
        .alias(
            "steps_since_initiator_transaction"
        ),

        pl.when(
            pl.col(
                "pair_previous_step"
            ).is_not_null()
        )
        .then(
            pl.col("step")
            -
            pl.col(
                "pair_previous_step"
            )
        )
        .otherwise(-1)
        .alias(
            "steps_since_pair_transaction"
        ),
    ])
)

# Fitur V3

In [35]:
# ============================================================
# 03.5 CANDIDATE V3 DERIVED BEHAVIORAL FEATURES
# ============================================================

behavioral_feature_lf = (
    behavioral_feature_lf
    .with_columns([
        (
            pl.col("amount_log1p")
            -
            pl.col("initiator_amount_mean_prior")
            .clip(lower_bound=0)
            .log1p()
        )
        .alias("amount_log_ratio_to_mean"),

        (
            pl.col("initiator_tx_count_6h")
            /
            (
                pl.col("initiator_tx_count_24h") / 4
                + 1
            )
        )
        .alias("tx_velocity_6h_vs_24h"),

        pl.when(
            pl.col("initiator_tx_count_6h") > 0
        )
        .then(
            pl.col("amount")
            /
            (
                pl.col("initiator_amount_sum_6h")
                /
                pl.col("initiator_tx_count_6h")
                + 1
            )
        )
        .otherwise(0)
        .alias("amount_vs_recent_mean_6h"),

        pl.when(
            pl.col("initiator_tx_count_24h") > 0
        )
        .then(
            pl.col("amount")
            /
            (
                pl.col("initiator_amount_sum_24h")
                /
                pl.col("initiator_tx_count_24h")
                + 1
            )
        )
        .otherwise(0)
        .alias("amount_vs_recent_mean_24h"),

        pl.col("recipient_tx_count_prior")
        .cast(pl.Float64)
        .log1p()
        .alias("recipient_history_log1p"),

        pl.col("pair_tx_count_prior")
        .cast(pl.Float64)
        .log1p()
        .alias("pair_history_log1p"),

        (
            (
                pl.col("steps_since_initiator_transaction")
                >= 0
            )
            &
            (
                pl.col("steps_since_initiator_transaction")
                <= 1
            )
        )
        .cast(pl.Int8)
        .alias("is_rapid_initiator_repeat"),

        (
            (
                pl.col("steps_since_pair_transaction")
                >= 0
            )
            &
            (
                pl.col("steps_since_pair_transaction")
                <= 1
            )
        )
        .cast(pl.Int8)
        .alias("is_rapid_pair_repeat"),

        (
            pl.col("new_bal_initiator")
            /
            (
                pl.col("old_bal_initiator").abs()
                + 1
            )
        )
        .alias("remaining_balance_ratio"),

        (
            pl.col("initiator_tx_count_prior") < 5
        )
        .cast(pl.Int8)
        .alias("is_cold_start"),
    ])
)

In [36]:
behavioral_preview = (
    behavioral_feature_lf
    .select([
        "step",
        "initiator",
        "recipient",
        "amount",
        "initiator_tx_count_prior",
        "initiator_amount_mean_prior",
        "amount_vs_initiator_mean",
        "initiator_tx_count_6h",
        "initiator_tx_count_24h",
        "recipient_tx_count_prior",
        "pair_tx_count_prior",
        "is_new_recipient",
        "target",
    ])
    .head(10)
    .collect()
)

display(behavioral_preview)

step,initiator,recipient,amount,initiator_tx_count_prior,initiator_amount_mean_prior,amount_vs_initiator_mean,initiator_tx_count_6h,initiator_tx_count_24h,recipient_tx_count_prior,pair_tx_count_prior,is_new_recipient,target
i64,str,str,f64,u32,f64,f64,u32,u32,u32,u32,i8,i8
0,"""4537027967639631""","""4875702729424478""",19824.96,0,0.0,0.0,0,0,0,0,1,1
0,"""4296267625767470""","""25-0000401""",598.97,0,0.0,0.0,0,0,0,0,1,0
0,"""4178224023847746""","""13-0001587""",545.85,0,0.0,0.0,0,0,0,0,1,0
0,"""4178224023847746""","""4096920916696293""",19847.01,0,0.0,0.0,0,0,0,0,1,1
0,"""4779013371563747""","""75-0003564""",546.89,0,0.0,0.0,0,0,0,0,1,0
0,"""4779013371563747""","""4593247015015219""",16973.75,0,0.0,0.0,0,0,0,0,1,1
0,"""4479448076398077""","""11-0003866""",659.02,0,0.0,0.0,0,0,0,0,1,0
0,"""4479448076398077""","""4649550475321678""",21103.99,0,0.0,0.0,0,0,0,0,1,1
0,"""4587942659105635""","""64-0008405""",648.89,0,0.0,0.0,0,0,0,0,1,0


In [37]:
behavioral_row_count = (
    behavioral_feature_lf
    .select(
        pl.len()
    )
    .collect(
        engine="streaming"
    )
    .item()
)

print(
    "Rows before behavioral features:",
    row_count,
)

print(
    "Rows after behavioral features:",
    behavioral_row_count,
)

if behavioral_row_count != row_count:
    raise ValueError(
        "Jumlah baris berubah setelah join behavioral features."
    )

Rows before behavioral features: 1720181
Rows after behavioral features: 1720181


In [38]:
# ============================================================
# AUDIT LABEL BERDASARKAN TIPE TRANSAKSI
# ============================================================

fraud_by_transaction_type_v3 = (
    behavioral_feature_lf
    .group_by("transaction_type")
    .agg([
        pl.len()
        .alias("transaction_count"),

        pl.col("target")
        .sum()
        .alias("fraud_count"),

        pl.col("target")
        .mean()
        .alias("fraud_rate"),
    ])
    .with_columns([
        (
            pl.col("transaction_count")
            - pl.col("fraud_count")
        )
        .alias("normal_count")
    ])
    .sort(
        "fraud_count",
        descending=True,
    )
    .collect(
        engine="streaming"
    )
)

display(
    fraud_by_transaction_type_v3
)

non_transfer_fraud_count_v3 = (
    behavioral_feature_lf
    .filter(
        (pl.col("transaction_type") != "TRANSFER")
        &
        (pl.col("target") == 1)
    )
    .select(
        pl.len()
    )
    .collect(
        engine="streaming"
    )
    .item()
)

print(
    "Fraud di luar TRANSFER:",
    non_transfer_fraud_count_v3,
)

if non_transfer_fraud_count_v3 > 0:
    raise ValueError(
        "Ditemukan fraud di luar TRANSFER. "
        "Filter global model perlu ditinjau ulang."
    )

transaction_type,transaction_count,fraud_count,fraud_rate,normal_count
str,u32,i64,f64,i64
"""TRANSFER""",569328,175518,0.30829,393810
"""DEPOSIT""",384431,0,0.0,384431
"""PAYMENT""",667245,0,0.0,667245
"""WITHDRAWAL""",93785,0,0.0,93785
"""DEBIT""",5392,0,0.0,5392


Fraud di luar TRANSFER: 0


In [39]:
# ============================================================
# TEMPORAL SPLIT CANDIDATE V3
# Train 70%
# Validation 10%
# Calibration 10%
# Test 10%
# ============================================================

unique_steps_v3 = (
    behavioral_feature_lf
    .select("step")
    .unique()
    .sort("step")
    .collect(
        engine="streaming"
    )
    .get_column("step")
)

number_of_steps_v3 = len(
    unique_steps_v3
)

if number_of_steps_v3 < 40:
    raise ValueError(
        "Jumlah step terlalu sedikit "
        "untuk empat temporal split."
    )

train_step_index_v3 = (
    int(
        number_of_steps_v3 * 0.70
    )
    - 1
)

validation_step_index_v3 = (
    int(
        number_of_steps_v3 * 0.80
    )
    - 1
)

calibration_step_index_v3 = (
    int(
        number_of_steps_v3 * 0.90
    )
    - 1
)

train_end_step_v3 = (
    unique_steps_v3[
        train_step_index_v3
    ]
)

validation_end_step_v3 = (
    unique_steps_v3[
        validation_step_index_v3
    ]
)

calibration_end_step_v3 = (
    unique_steps_v3[
        calibration_step_index_v3
    ]
)

print(
    "Jumlah step:",
    number_of_steps_v3,
)

print(
    "Train:",
    unique_steps_v3[0],
    "sampai",
    train_end_step_v3,
)

print(
    "Validation:",
    train_end_step_v3 + 1,
    "sampai",
    validation_end_step_v3,
)

print(
    "Calibration:",
    validation_end_step_v3 + 1,
    "sampai",
    calibration_end_step_v3,
)

print(
    "Test:",
    calibration_end_step_v3 + 1,
    "sampai",
    unique_steps_v3[-1],
)

Jumlah step: 144
Train: 0 sampai 99
Validation: 100 sampai 114
Calibration: 115 sampai 128
Test: 129 sampai 143


In [40]:
# ============================================================
# GLOBAL MODEL V3
# HANYA TRANSAKSI TRANSFER
# ============================================================

global_model_filter_v3 = (
    pl.col("transaction_type")
    == "TRANSFER"
)

train_v3_lf = (
    behavioral_feature_lf
    .filter(
        global_model_filter_v3
        &
        (
            pl.col("step")
            <= train_end_step_v3
        )
    )
)

validation_v3_lf = (
    behavioral_feature_lf
    .filter(
        global_model_filter_v3
        &
        (
            pl.col("step")
            > train_end_step_v3
        )
        &
        (
            pl.col("step")
            <= validation_end_step_v3
        )
    )
)

calibration_v3_lf = (
    behavioral_feature_lf
    .filter(
        global_model_filter_v3
        &
        (
            pl.col("step")
            > validation_end_step_v3
        )
        &
        (
            pl.col("step")
            <= calibration_end_step_v3
        )
    )
)

test_v3_lf = (
    behavioral_feature_lf
    .filter(
        global_model_filter_v3
        &
        (
            pl.col("step")
            > calibration_end_step_v3
        )
    )
)

In [41]:
# ============================================================
# RINGKASAN TEMPORAL SPLIT V3
# ============================================================

def summarize_split_v3(
    split_name,
    split_lf,
):
    return (
        split_lf
        .select([
            pl.lit(
                split_name
            )
            .alias("split"),

            pl.len()
            .alias("row_count"),

            pl.col("step")
            .min()
            .alias("step_start"),

            pl.col("step")
            .max()
            .alias("step_end"),

            pl.col("target")
            .sum()
            .alias("fraud_count"),

            (
                pl.len()
                - pl.col("target").sum()
            )
            .alias("normal_count"),

            pl.col("target")
            .mean()
            .alias("fraud_rate"),

            pl.col("initiator")
            .n_unique()
            .alias("unique_initiators"),

            pl.col("recipient")
            .n_unique()
            .alias("unique_recipients"),
        ])
        .collect(
            engine="streaming"
        )
    )


split_summary_v3 = pl.concat([
    summarize_split_v3(
        "train",
        train_v3_lf,
    ),
    summarize_split_v3(
        "validation",
        validation_v3_lf,
    ),
    summarize_split_v3(
        "calibration",
        calibration_v3_lf,
    ),
    summarize_split_v3(
        "test",
        test_v3_lf,
    ),
])

display(
    split_summary_v3
)

for split_row in split_summary_v3.iter_rows(
    named=True
):
    if split_row["row_count"] == 0:
        raise ValueError(
            f"Split {split_row['split']} kosong."
        )

    if split_row["fraud_count"] == 0:
        raise ValueError(
            f"Split {split_row['split']} "
            "tidak memiliki fraud."
        )

    if split_row["normal_count"] == 0:
        raise ValueError(
            f"Split {split_row['split']} "
            "tidak memiliki transaksi normal."
        )

split,row_count,step_start,step_end,fraud_count,normal_count,fraud_rate,unique_initiators,unique_recipients
str,u32,i64,i64,i64,i64,f64,u32,u32
"""train""",391716,0,99,122258,269458,0.312109,10863,9593
"""validation""",58895,100,114,18208,40687,0.30916,7629,5000
"""calibration""",55066,115,128,16915,38151,0.307177,7522,5190
"""test""",63651,129,143,18137,45514,0.284944,7946,7708


In [42]:
# ============================================================
# PATH DATASET CANDIDATE V3
# ============================================================

TRAIN_V3_PATH = (
    PROCESSED_DATA_DIR
    / "momtsim_train_features_v3.parquet"
)

VALIDATION_V3_PATH = (
    PROCESSED_DATA_DIR
    / "momtsim_validation_features_v3.parquet"
)

CALIBRATION_V3_PATH = (
    PROCESSED_DATA_DIR
    / "momtsim_calibration_features_v3.parquet"
)

TEST_V3_PATH = (
    PROCESSED_DATA_DIR
    / "momtsim_test_features_v3.parquet"
)

print(
    "Train path:",
    TRAIN_V3_PATH,
)

print(
    "Validation path:",
    VALIDATION_V3_PATH,
)

print(
    "Calibration path:",
    CALIBRATION_V3_PATH,
)

print(
    "Test path:",
    TEST_V3_PATH,
)

Train path: /content/drive/MyDrive/favera/model/transaction-anomaly/data/processed/momtsim_train_features_v3.parquet
Validation path: /content/drive/MyDrive/favera/model/transaction-anomaly/data/processed/momtsim_validation_features_v3.parquet
Calibration path: /content/drive/MyDrive/favera/model/transaction-anomaly/data/processed/momtsim_calibration_features_v3.parquet
Test path: /content/drive/MyDrive/favera/model/transaction-anomaly/data/processed/momtsim_test_features_v3.parquet


In [43]:
# ============================================================
# SIMPAN TEMPORAL SPLIT V3
# ============================================================

train_v3_lf.sink_parquet(
    str(TRAIN_V3_PATH),
    compression="zstd",
)

validation_v3_lf.sink_parquet(
    str(VALIDATION_V3_PATH),
    compression="zstd",
)

calibration_v3_lf.sink_parquet(
    str(CALIBRATION_V3_PATH),
    compression="zstd",
)

test_v3_lf.sink_parquet(
    str(TEST_V3_PATH),
    compression="zstd",
)

print(
    "Train V3 tersimpan:",
    TRAIN_V3_PATH,
)

print(
    "Validation V3 tersimpan:",
    VALIDATION_V3_PATH,
)

print(
    "Calibration V3 tersimpan:",
    CALIBRATION_V3_PATH,
)

print(
    "Test V3 tersimpan:",
    TEST_V3_PATH,
)

Train V3 tersimpan: /content/drive/MyDrive/favera/model/transaction-anomaly/data/processed/momtsim_train_features_v3.parquet
Validation V3 tersimpan: /content/drive/MyDrive/favera/model/transaction-anomaly/data/processed/momtsim_validation_features_v3.parquet
Calibration V3 tersimpan: /content/drive/MyDrive/favera/model/transaction-anomaly/data/processed/momtsim_calibration_features_v3.parquet
Test V3 tersimpan: /content/drive/MyDrive/favera/model/transaction-anomaly/data/processed/momtsim_test_features_v3.parquet


In [44]:
del base_history_df
del initiator_step_df
del initiator_6h_df
del initiator_24h_df
del recipient_step_df
del pair_step_df

gc.collect()

print("Intermediate memory dibersihkan.")

Intermediate memory dibersihkan.


In [45]:
# ============================================================
# FEATURE CONTRACT CANDIDATE V3
# ============================================================

V3_NUMERIC_FEATURES = [
    # --------------------------------------------------------
    # Nominal transaksi
    # --------------------------------------------------------
    "amount",
    "amount_log1p",

    # --------------------------------------------------------
    # Kondisi saldo pengirim
    # Fitur saldo penerima sengaja tidak digunakan
    # karena umumnya tidak tersedia dalam e-statement.
    # --------------------------------------------------------
    "old_bal_initiator",
    "new_bal_initiator",
    "initiator_balance_change",
    "amount_to_initiator_balance_ratio",
    "initiator_balance_drop_ratio",
    "initiator_old_balance_non_positive",
    "initiator_new_balance_negative",
    "initiator_balance_error",
    "remaining_balance_ratio",

    # --------------------------------------------------------
    # Waktu
    # 1 step MoMTSim = 1 jam
    # --------------------------------------------------------
    "hour_of_day",
    "is_night",
    "hour_sin",
    "hour_cos",

    # --------------------------------------------------------
    # Histori pengirim
    # --------------------------------------------------------
    "has_initiator_history",
    "initiator_tx_count_prior",
    "initiator_amount_mean_prior",
    "initiator_amount_std_prior",
    "initiator_amount_max_prior",
    "amount_vs_initiator_mean",
    "amount_zscore_initiator",
    "amount_vs_initiator_max",
    "amount_log_ratio_to_mean",

    # --------------------------------------------------------
    # Aktivitas terbaru
    # --------------------------------------------------------
    "initiator_tx_count_6h",
    "initiator_amount_sum_6h",
    "initiator_tx_count_24h",
    "initiator_amount_sum_24h",
    "tx_velocity_6h_vs_24h",
    "amount_vs_recent_mean_6h",
    "amount_vs_recent_mean_24h",
    "steps_since_initiator_transaction",
    "is_rapid_initiator_repeat",

    # --------------------------------------------------------
    # Histori penerima dan pasangan
    # --------------------------------------------------------
    "recipient_tx_count_prior",
    "recipient_history_log1p",
    "pair_tx_count_prior",
    "pair_history_log1p",
    "is_new_recipient",
    "steps_since_pair_transaction",
    "is_rapid_pair_repeat",

    # --------------------------------------------------------
    # Cold-start
    # --------------------------------------------------------
    "is_cold_start",
]

# Kolom ini tidak masuk ke model.
# Kolom dipertahankan untuk audit hasil prediksi.
V3_AUDIT_COLUMNS = [
    "step",
    "initiator",
    "recipient",
    "transaction_type",
]

V3_TARGET = "target"

print(
    "Jumlah fitur numerik V3:",
    len(V3_NUMERIC_FEATURES),
)

Jumlah fitur numerik V3: 41


In [46]:
# ============================================================
# LOADER DATASET CANDIDATE V3
# ============================================================

def load_v3_split(path):
    selected_columns = (
        V3_AUDIT_COLUMNS
        + V3_NUMERIC_FEATURES
        + [V3_TARGET]
    )

    lazy_split = pl.scan_parquet(
        str(path)
    )

    available_columns = (
        lazy_split
        .collect_schema()
        .names()
    )

    missing_columns = [
        column
        for column in selected_columns
        if column not in available_columns
    ]

    if missing_columns:
        raise ValueError(
            f"Kolom V3 tidak ditemukan pada {path}: "
            f"{missing_columns}"
        )

    numeric_expressions = [
        pl.col(column)
        .cast(
            pl.Float32,
            strict=False,
        )
        .fill_nan(0)
        .fill_null(0)
        .alias(column)
        for column in V3_NUMERIC_FEATURES
    ]

    loaded_df = (
        lazy_split
        .select(selected_columns)
        .with_columns(
            numeric_expressions
            + [
                pl.col("step")
                .cast(pl.Int64),

                pl.col("initiator")
                .cast(pl.String)
                .fill_null("UNKNOWN"),

                pl.col("recipient")
                .cast(pl.String)
                .fill_null("UNKNOWN"),

                pl.col("transaction_type")
                .cast(pl.String)
                .fill_null("UNKNOWN"),

                pl.col(V3_TARGET)
                .cast(pl.Int8)
                .alias(V3_TARGET),
            ]
        )
        .collect(
            engine="streaming"
        )
        .to_pandas()
    )

    return loaded_df

In [47]:
# ============================================================
# LOAD EMPAT TEMPORAL SPLIT V3
# ============================================================

train_v3_df = load_v3_split(
    TRAIN_V3_PATH
)

validation_v3_df = load_v3_split(
    VALIDATION_V3_PATH
)

calibration_v3_df = load_v3_split(
    CALIBRATION_V3_PATH
)

test_v3_df = load_v3_split(
    TEST_V3_PATH
)

print(
    "Train V3:",
    train_v3_df.shape,
)

print(
    "Validation V3:",
    validation_v3_df.shape,
)

print(
    "Calibration V3:",
    calibration_v3_df.shape,
)

print(
    "Test V3:",
    test_v3_df.shape,
)

Train V3: (391716, 46)
Validation V3: (58895, 46)
Calibration V3: (55066, 46)
Test V3: (63651, 46)


In [48]:
# ============================================================
# AUDIT DATASET YANG SUDAH DIMUAT
# ============================================================

loaded_split_summary_v3 = pd.DataFrame([
    {
        "split": "train",
        "rows": len(train_v3_df),
        "fraud_count": int(
            train_v3_df[V3_TARGET].sum()
        ),
        "fraud_rate": float(
            train_v3_df[V3_TARGET].mean()
        ),
        "step_start": int(
            train_v3_df["step"].min()
        ),
        "step_end": int(
            train_v3_df["step"].max()
        ),
    },
    {
        "split": "validation",
        "rows": len(validation_v3_df),
        "fraud_count": int(
            validation_v3_df[V3_TARGET].sum()
        ),
        "fraud_rate": float(
            validation_v3_df[V3_TARGET].mean()
        ),
        "step_start": int(
            validation_v3_df["step"].min()
        ),
        "step_end": int(
            validation_v3_df["step"].max()
        ),
    },
    {
        "split": "calibration",
        "rows": len(calibration_v3_df),
        "fraud_count": int(
            calibration_v3_df[V3_TARGET].sum()
        ),
        "fraud_rate": float(
            calibration_v3_df[V3_TARGET].mean()
        ),
        "step_start": int(
            calibration_v3_df["step"].min()
        ),
        "step_end": int(
            calibration_v3_df["step"].max()
        ),
    },
    {
        "split": "test",
        "rows": len(test_v3_df),
        "fraud_count": int(
            test_v3_df[V3_TARGET].sum()
        ),
        "fraud_rate": float(
            test_v3_df[V3_TARGET].mean()
        ),
        "step_start": int(
            test_v3_df["step"].min()
        ),
        "step_end": int(
            test_v3_df["step"].max()
        ),
    },
])

display(
    loaded_split_summary_v3
)

for split_name, split_df in [
    ("train", train_v3_df),
    ("validation", validation_v3_df),
    ("calibration", calibration_v3_df),
    ("test", test_v3_df),
]:
    transaction_types = (
        split_df["transaction_type"]
        .dropna()
        .unique()
        .tolist()
    )

    print(
        split_name,
        "transaction types:",
        transaction_types,
    )

    if transaction_types != ["TRANSFER"]:
        raise ValueError(
            f"Split {split_name} masih memiliki "
            f"tipe selain TRANSFER: "
            f"{transaction_types}"
        )

,split,rows,fraud_count,fraud_rate,step_start,step_end
0,train,391716,122258,0.312109,0,99
1,validation,58895,18208,0.309160,100,114
2,calibration,55066,16915,0.307177,115,128
3,test,63651,18137,0.284944,129,143


train transaction types: ['TRANSFER']
validation transaction types: ['TRANSFER']
calibration transaction types: ['TRANSFER']
test transaction types: ['TRANSFER']


In [49]:
# ============================================================
# MEMBENTUK MATRIKS MODEL V3
# ============================================================

def build_model_matrix_v3(
    dataframe,
):
    X = (
        dataframe[
            V3_NUMERIC_FEATURES
        ]
        .copy()
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .fillna(0)
        .astype("float32")
    )

    y = (
        dataframe[V3_TARGET]
        .astype("int8")
        .copy()
    )

    audit = (
        dataframe[
            V3_AUDIT_COLUMNS
        ]
        .copy()
    )

    return X, y, audit


(
    X_train_v3,
    y_train_v3,
    train_audit_v3,
) = build_model_matrix_v3(
    train_v3_df
)

(
    X_validation_v3,
    y_validation_v3,
    validation_audit_v3,
) = build_model_matrix_v3(
    validation_v3_df
)

(
    X_calibration_v3,
    y_calibration_v3,
    calibration_audit_v3,
) = build_model_matrix_v3(
    calibration_v3_df
)

(
    X_test_v3,
    y_test_v3,
    test_audit_v3,
) = build_model_matrix_v3(
    test_v3_df
)

In [50]:
# ============================================================
# VALIDASI MATRIKS MODEL V3
# ============================================================

matrix_summary_v3 = pd.DataFrame([
    {
        "split": "train",
        "rows": X_train_v3.shape[0],
        "features": X_train_v3.shape[1],
        "fraud_rate": y_train_v3.mean(),
        "missing_values": int(
            X_train_v3
            .isna()
            .sum()
            .sum()
        ),
        "infinite_values": int(
            np.isinf(
                X_train_v3.to_numpy()
            ).sum()
        ),
    },
    {
        "split": "validation",
        "rows": X_validation_v3.shape[0],
        "features": X_validation_v3.shape[1],
        "fraud_rate": y_validation_v3.mean(),
        "missing_values": int(
            X_validation_v3
            .isna()
            .sum()
            .sum()
        ),
        "infinite_values": int(
            np.isinf(
                X_validation_v3.to_numpy()
            ).sum()
        ),
    },
    {
        "split": "calibration",
        "rows": X_calibration_v3.shape[0],
        "features": X_calibration_v3.shape[1],
        "fraud_rate": y_calibration_v3.mean(),
        "missing_values": int(
            X_calibration_v3
            .isna()
            .sum()
            .sum()
        ),
        "infinite_values": int(
            np.isinf(
                X_calibration_v3.to_numpy()
            ).sum()
        ),
    },
    {
        "split": "test",
        "rows": X_test_v3.shape[0],
        "features": X_test_v3.shape[1],
        "fraud_rate": y_test_v3.mean(),
        "missing_values": int(
            X_test_v3
            .isna()
            .sum()
            .sum()
        ),
        "infinite_values": int(
            np.isinf(
                X_test_v3.to_numpy()
            ).sum()
        ),
    },
])

display(
    matrix_summary_v3
)

assert list(X_train_v3.columns) == V3_NUMERIC_FEATURES
assert list(X_validation_v3.columns) == V3_NUMERIC_FEATURES
assert list(X_calibration_v3.columns) == V3_NUMERIC_FEATURES
assert list(X_test_v3.columns) == V3_NUMERIC_FEATURES

assert (
    matrix_summary_v3["missing_values"]
    == 0
).all()

assert (
    matrix_summary_v3["infinite_values"]
    == 0
).all()

print(
    "Matriks Candidate V3 siap untuk training."
)

,split,rows,features,fraud_rate,missing_values,infinite_values
0,train,391716,41,0.312109,0,0
1,validation,58895,41,0.309160,0,0
2,calibration,55066,41,0.307177,0,0
3,test,63651,41,0.284944,0,0


Matriks Candidate V3 siap untuk training.


In [51]:
# ============================================================
# 07.3 CANDIDATE V4 THRESHOLD AND EVALUATION UTILITIES
# ============================================================

def select_threshold_v4(
    model_name,
    y_true,
    probabilities,
):
    """
    Memilih threshold hanya dari calibration set.

    Prioritas:
    1. Memenuhi seluruh target.
    2. Jika gagal, pertahankan recall minimum.
    3. Pada recall minimum, pilih FPR terendah.
    """
    operating_points = build_operating_point_table(
        y_true=y_true,
        probabilities=probabilities,
    )

    pr_auc = float(
        average_precision_score(
            y_true,
            probabilities,
        )
    )

    feasible_points = operating_points[
        (
            operating_points["precision"]
            >= TARGET_PRECISION
        )
        &
        (
            operating_points["recall"]
            >= TARGET_RECALL
        )
        &
        (
            operating_points["f1"]
            >= TARGET_F1
        )
        &
        (
            operating_points["false_positive_rate"]
            <= TARGET_FPR
        )
    ].copy()

    if not feasible_points.empty:
        selected = (
            feasible_points
            .sort_values(
                [
                    "f1",
                    "false_positive_rate",
                    "precision",
                    "threshold",
                ],
                ascending=[
                    False,
                    True,
                    False,
                    False,
                ],
            )
            .iloc[0]
        )

        selection_strategy = "ALL_TARGETS_ACHIEVED"

    else:
        recall_guardrail = operating_points[
            operating_points["recall"]
            >= TARGET_RECALL
        ].copy()

        if not recall_guardrail.empty:
            selected = (
                recall_guardrail
                .sort_values(
                    [
                        "false_positive_rate",
                        "precision",
                        "f1",
                        "threshold",
                    ],
                    ascending=[
                        True,
                        False,
                        False,
                        False,
                    ],
                )
                .iloc[0]
            )

            selection_strategy = (
                "MINIMUM_FPR_AT_RECALL_GUARDRAIL"
            )

        else:
            selected = (
                operating_points
                .sort_values(
                    [
                        "f1",
                        "precision",
                        "recall",
                    ],
                    ascending=[
                        False,
                        False,
                        False,
                    ],
                )
                .iloc[0]
            )

            selection_strategy = "BEST_AVAILABLE_F1"

    target_achieved = (
        pr_auc >= TARGET_PR_AUC
        and selected["precision"] >= TARGET_PRECISION
        and selected["recall"] >= TARGET_RECALL
        and selected["f1"] >= TARGET_F1
        and selected["false_positive_rate"] <= TARGET_FPR
    )

    summary = pd.DataFrame([
        {
            "model": model_name,
            "status": (
                "TARGET ACHIEVED"
                if target_achieved
                else "TARGET NOT ACHIEVED"
            ),
            "selection_strategy": selection_strategy,
            "pr_auc": pr_auc,
            "threshold": float(
                selected["threshold"]
            ),
            "precision": float(
                selected["precision"]
            ),
            "recall": float(
                selected["recall"]
            ),
            "f1": float(
                selected["f1"]
            ),
            "false_positive_rate": float(
                selected["false_positive_rate"]
            ),
            "true_positive": int(
                selected["true_positive"]
            ),
            "false_positive": int(
                selected["false_positive"]
            ),
            "false_negative": int(
                selected["false_negative"]
            ),
            "true_negative": int(
                selected["true_negative"]
            ),
        }
    ])

    return summary, operating_points


def evaluate_fixed_threshold_v4(
    model_name,
    y_true,
    probabilities,
    threshold,
):
    """
    Mengevaluasi probabilitas dengan threshold terkunci.
    """
    y_true = np.asarray(
        y_true,
        dtype=np.int8,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    predictions = (
        probabilities >= threshold
    ).astype(np.int8)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    ).ravel()

    precision = (
        tp / max(tp + fp, 1)
    )

    recall = (
        tp / max(tp + fn, 1)
    )

    f1 = (
        2 * precision * recall
        / max(
            precision + recall,
            1e-12,
        )
    )

    false_positive_rate = (
        fp / max(fp + tn, 1)
    )

    fraud_rate = float(
        y_true.mean()
    )

    pr_auc = float(
        average_precision_score(
            y_true,
            probabilities,
        )
    )

    target_achieved = (
        pr_auc >= TARGET_PR_AUC
        and precision >= TARGET_PRECISION
        and recall >= TARGET_RECALL
        and f1 >= TARGET_F1
        and false_positive_rate <= TARGET_FPR
    )

    return pd.DataFrame([
        {
            "model": model_name,
            "status": (
                "TARGET ACHIEVED"
                if target_achieved
                else "TARGET NOT ACHIEVED"
            ),
            "rows": len(y_true),
            "fraud_rate": fraud_rate,
            "pr_auc": pr_auc,
            "threshold": float(threshold),
            "precision": float(precision),
            "precision_lift": float(
                precision
                / max(fraud_rate, 1e-12)
            ),
            "recall": float(recall),
            "f1": float(f1),
            "false_positive_rate": float(
                false_positive_rate
            ),
            "alert_rate": float(
                predictions.mean()
            ),
            "true_positive": int(tp),
            "false_positive": int(fp),
            "false_negative": int(fn),
            "true_negative": int(tn),
        }
    ])

In [52]:
# ============================================================
# 08.0 CANDIDATE V4 FEATURE DIAGNOSTIC
# ============================================================

validation_base_rate_v4 = float(
    y_validation_v3.mean()
)

feature_diagnostic_rows_v4 = []

for feature in V3_NUMERIC_FEATURES:
    train_series = X_train_v3[feature]
    validation_series = X_validation_v3[feature]

    train_unique = int(
        train_series.nunique(
            dropna=False
        )
    )

    validation_unique = int(
        validation_series.nunique(
            dropna=False
        )
    )

    train_std = float(
        train_series.std(
            ddof=0
        )
    )

    validation_std = float(
        validation_series.std(
            ddof=0
        )
    )

    train_zero_rate = float(
        np.isclose(
            train_series.to_numpy(),
            0.0,
        ).mean()
    )

    validation_zero_rate = float(
        np.isclose(
            validation_series.to_numpy(),
            0.0,
        ).mean()
    )

    is_constant_train = (
        train_unique <= 1
    )

    is_constant_validation = (
        validation_unique <= 1
    )

    if (
        is_constant_train
        or is_constant_validation
    ):
        best_univariate_ap = (
            validation_base_rate_v4
        )

        positive_ap = np.nan
        negative_ap = np.nan

        if (
            is_constant_train
            and is_constant_validation
        ):
            risk_direction = (
                "constant_train_and_validation"
            )

        elif is_constant_train:
            risk_direction = (
                "constant_train"
            )

        else:
            risk_direction = (
                "constant_validation"
            )

    else:
        positive_ap = float(
            average_precision_score(
                y_validation_v3,
                validation_series,
            )
        )

        negative_ap = float(
            average_precision_score(
                y_validation_v3,
                -validation_series,
            )
        )

        if positive_ap >= negative_ap:
            best_univariate_ap = (
                positive_ap
            )

            risk_direction = (
                "higher_is_risk"
            )

        else:
            best_univariate_ap = (
                negative_ap
            )

            risk_direction = (
                "lower_is_risk"
            )

    validation_ap_lift = float(
        best_univariate_ap
        / max(
            validation_base_rate_v4,
            1e-12,
        )
    )

    feature_diagnostic_rows_v4.append({
        "feature": feature,
        "train_unique": (
            train_unique
        ),
        "validation_unique": (
            validation_unique
        ),
        "train_std": (
            train_std
        ),
        "validation_std": (
            validation_std
        ),
        "train_zero_rate": (
            train_zero_rate
        ),
        "validation_zero_rate": (
            validation_zero_rate
        ),
        "positive_univariate_ap": (
            positive_ap
        ),
        "negative_univariate_ap": (
            negative_ap
        ),
        "validation_univariate_ap": (
            best_univariate_ap
        ),
        "validation_ap_lift": (
            validation_ap_lift
        ),
        "is_constant_train": (
            is_constant_train
        ),
        "is_constant_validation": (
            is_constant_validation
        ),
        "risk_direction": (
            risk_direction
        ),
    })

feature_diagnostic_v4 = (
    pd.DataFrame(
        feature_diagnostic_rows_v4
    )
    .sort_values(
        [
            "validation_univariate_ap",
            "train_unique",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)

constant_features_v4 = (
    feature_diagnostic_v4.loc[
        feature_diagnostic_v4[
            "train_unique"
        ] <= 1,
        "feature",
    ]
    .tolist()
)

non_generalizing_features_v4 = (
    feature_diagnostic_v4.loc[
        (
            feature_diagnostic_v4[
                "train_unique"
            ] > 1
        )
        &
        (
            feature_diagnostic_v4[
                "validation_unique"
            ] <= 1
        ),
        "feature",
    ]
    .tolist()
)

constant_validation_features_v4 = (
    feature_diagnostic_v4.loc[
        feature_diagnostic_v4[
            "validation_unique"
        ] <= 1,
        "feature",
    ]
    .tolist()
)

weak_signal_features_v4 = (
    feature_diagnostic_v4.loc[
        feature_diagnostic_v4[
            "validation_ap_lift"
        ] <= 1.01,
        "feature",
    ]
    .tolist()
)

strongest_features_v4 = (
    feature_diagnostic_v4.loc[
        feature_diagnostic_v4[
            "validation_ap_lift"
        ] > 1.05,
        [
            "feature",
            "validation_univariate_ap",
            "validation_ap_lift",
            "risk_direction",
        ],
    ]
    .copy()
)

display(
    feature_diagnostic_v4
)

print(
    "Validation fraud rate:",
    validation_base_rate_v4,
)

print(
    "Jumlah fitur konstan pada train:",
    len(
        constant_features_v4
    ),
)

print(
    "Fitur konstan pada train:",
    constant_features_v4,
)

print(
    "Jumlah fitur konstan pada validation:",
    len(
        constant_validation_features_v4
    ),
)

print(
    "Fitur konstan pada validation:",
    constant_validation_features_v4,
)

print(
    "Jumlah fitur berubah pada train "
    "tetapi konstan pada validation:",
    len(
        non_generalizing_features_v4
    ),
)

print(
    "Fitur tidak bergeneralisasi:",
    non_generalizing_features_v4,
)

print(
    "Jumlah fitur dengan AP lift <= 1.01:",
    len(
        weak_signal_features_v4
    ),
)

print(
    "Fitur dengan sinyal univariat lemah:",
    weak_signal_features_v4,
)

print(
    "Fitur dengan AP lift > 1.05:"
)

display(
    strongest_features_v4
)

,feature,train_unique,validation_unique,train_std,validation_std,train_zero_rate,validation_zero_rate,positive_univariate_ap,negative_univariate_ap,validation_univariate_ap,validation_ap_lift,is_constant_train,is_constant_validation,risk_direction
0,initiator_amount_max_prior,24106,7392,3.145496e+05,3.041305e+05,0.013520,0.000000,0.303095,0.359710,0.359710,1.163506,False,False,lower_is_risk
1,amount_vs_initiator_max,253676,50979,5.048113e+00,8.863457e-02,0.013520,0.000000,0.358876,0.302896,0.358876,1.160810,False,False,higher_is_risk
2,initiator_amount_std_prior,254274,39897,3.206993e+04,2.309033e+04,0.017120,0.000085,0.301185,0.357704,0.357704,1.157017,False,False,lower_is_risk
3,initiator_amount_mean_prior,256178,39913,3.105401e+04,1.890128e+04,0.013520,0.000000,0.299267,0.357545,0.357545,1.156504,False,False,lower_is_risk
4,amount_vs_initiator_mean,380172,58247,5.296350e+00,3.751280e-01,0.013520,0.000000,0.357444,0.299512,0.357444,1.156177,False,False,higher_is_risk
5,amount_log_ratio_to_mean,380200,58251,1.515606e+00,4.910444e-01,0.000000,0.000000,0.357444,0.299512,0.357444,1.156176,False,False,higher_is_risk
6,amount_zscore_initiator,378765,58232,1.931609e+02,2.770151e-01,0.017112,0.000085,0.357218,0.299436,0.357218,1.155444,False,False,higher_is_risk
7,initiator_amount_sum_24h,255961,39675,4.447506e+06,1.394348e+06,0.016823,0.004686,0.300427,0.337397,0.337397,1.091334,False,False,lower_is_risk
8,amount_vs_recent_mean_24h,378790,57959,5.610505e+00,1.414104e+00,0.016823,0.004686,0.336298,0.303584,0.336298,1.087779,False,False,higher_is_risk
9,initiator_tx_count_24h,442,245,5.495603e+01,3.997535e+01,0.016823,0.004686,0.300934,0.328521,0.328521,1.062622,False,False,lower_is_risk


Validation fraud rate: 0.3091603701502674
Jumlah fitur konstan pada train: 0
Fitur konstan pada train: []
Jumlah fitur konstan pada validation: 1
Fitur konstan pada validation: ['has_initiator_history']
Jumlah fitur berubah pada train tetapi konstan pada validation: 1
Fitur tidak bergeneralisasi: ['has_initiator_history']
Jumlah fitur dengan AP lift <= 1.01: 15
Fitur dengan sinyal univariat lemah: ['amount', 'amount_log1p', 'hour_sin', 'steps_since_pair_transaction', 'pair_tx_count_prior', 'pair_history_log1p', 'hour_of_day', 'hour_cos', 'is_new_recipient', 'initiator_new_balance_negative', 'initiator_old_balance_non_positive', 'is_night', 'is_rapid_pair_repeat', 'is_cold_start', 'has_initiator_history']
Fitur dengan AP lift > 1.05:


,feature,validation_univariate_ap,validation_ap_lift,risk_direction
0,initiator_amount_max_prior,0.359710,1.163506,lower_is_risk
1,amount_vs_initiator_max,0.358876,1.160810,higher_is_risk
2,initiator_amount_std_prior,0.357704,1.157017,lower_is_risk
3,initiator_amount_mean_prior,0.357545,1.156504,lower_is_risk
4,amount_vs_initiator_mean,0.357444,1.156177,higher_is_risk
5,amount_log_ratio_to_mean,0.357444,1.156176,higher_is_risk
6,amount_zscore_initiator,0.357218,1.155444,higher_is_risk
7,initiator_amount_sum_24h,0.337397,1.091334,lower_is_risk
8,amount_vs_recent_mean_24h,0.336298,1.087779,higher_is_risk
9,initiator_tx_count_24h,0.328521,1.062622,lower_is_risk


In [53]:
# ============================================================
# 08.1 CANDIDATE V4 DEPLOYABLE INTERACTION FEATURES
# ============================================================

V4_DERIVED_FEATURES = [
    "initiator_history_log1p",
    "amount_log_ratio_to_prior_max",
    "amount_excess_over_prior_max_ratio",
    "is_amount_above_prior_max",
    "pair_share_of_initiator_history",
    "current_amount_share_6h",
    "current_amount_share_24h",
    "recent_sum_6h_share_24h",
    "steps_since_initiator_log1p",
    "steps_since_pair_log1p",
    "new_recipient_amount_log",
    "cold_start_amount_log",
    "initiator_balance_error_log1p",
    "is_balance_nearly_depleted",
    "amount_to_balance_ratio_capped",
]


def add_v4_derived_features(
    dataframe,
):
    output = dataframe.copy()

    amount = (
        output["amount"]
        .clip(lower=0)
    )

    amount_log = np.log1p(
        amount
    )

    prior_max = (
        output[
            "initiator_amount_max_prior"
        ]
        .clip(lower=0)
    )

    has_history = (
        output[
            "has_initiator_history"
        ] > 0
    ).astype("float32")

    history_count = (
        output[
            "initiator_tx_count_prior"
        ]
        .clip(lower=0)
    )

    output[
        "initiator_history_log1p"
    ] = np.log1p(
        history_count
    )

    output[
        "amount_log_ratio_to_prior_max"
    ] = (
        amount_log
        - np.log1p(prior_max)
    ) * has_history

    output[
        "amount_excess_over_prior_max_ratio"
    ] = (
        (
            amount - prior_max
        )
        .clip(lower=0)
        /
        (
            prior_max + 1.0
        )
    ) * has_history

    output[
        "is_amount_above_prior_max"
    ] = (
        (
            output[
                "has_initiator_history"
            ] > 0
        )
        &
        (
            amount > prior_max
        )
    ).astype("float32")

    output[
        "pair_share_of_initiator_history"
    ] = (
        output[
            "pair_tx_count_prior"
        ]
        /
        (
            history_count + 1.0
        )
    )

    output[
        "current_amount_share_6h"
    ] = (
        amount
        /
        (
            output[
                "initiator_amount_sum_6h"
            ]
            + amount
            + 1.0
        )
    )

    output[
        "current_amount_share_24h"
    ] = (
        amount
        /
        (
            output[
                "initiator_amount_sum_24h"
            ]
            + amount
            + 1.0
        )
    )

    output[
        "recent_sum_6h_share_24h"
    ] = (
        output[
            "initiator_amount_sum_6h"
        ]
        /
        (
            output[
                "initiator_amount_sum_24h"
            ]
            + 1.0
        )
    )

    output[
        "steps_since_initiator_log1p"
    ] = np.where(
        output[
            "steps_since_initiator_transaction"
        ] >= 0,
        np.log1p(
            output[
                "steps_since_initiator_transaction"
            ].clip(lower=0)
        ),
        0.0,
    )

    output[
        "steps_since_pair_log1p"
    ] = np.where(
        output[
            "steps_since_pair_transaction"
        ] >= 0,
        np.log1p(
            output[
                "steps_since_pair_transaction"
            ].clip(lower=0)
        ),
        0.0,
    )

    output[
        "new_recipient_amount_log"
    ] = (
        output["is_new_recipient"]
        * amount_log
    )

    output[
        "cold_start_amount_log"
    ] = (
        output["is_cold_start"]
        * amount_log
    )

    output[
        "initiator_balance_error_log1p"
    ] = np.log1p(
        output[
            "initiator_balance_error"
        ].clip(lower=0)
    )

    old_balance = (
        output["old_bal_initiator"]
        .abs()
    )

    remaining_ratio = (
        output["new_bal_initiator"]
        /
        (
            old_balance + 1.0
        )
    )

    output[
        "is_balance_nearly_depleted"
    ] = (
        (
            output[
                "old_bal_initiator"
            ] > 0
        )
        &
        (
            remaining_ratio <= 0.05
        )
    ).astype("float32")

    output[
        "amount_to_balance_ratio_capped"
    ] = (
        output[
            "amount_to_initiator_balance_ratio"
        ]
        .clip(
            lower=0,
            upper=20,
        )
    )

    output = (
        output
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .fillna(0)
    )

    output[
        V4_DERIVED_FEATURES
    ] = (
        output[
            V4_DERIVED_FEATURES
        ]
        .astype("float32")
    )

    return output


V4_BASE_FEATURES = [
    feature
    for feature in V3_NUMERIC_FEATURES
    if feature not in constant_features_v4
]

V4_NUMERIC_FEATURES = (
    V4_BASE_FEATURES
    + V4_DERIVED_FEATURES
)

X_train_v4 = (
    add_v4_derived_features(
        X_train_v3
    )[V4_NUMERIC_FEATURES]
    .astype("float32")
)

X_validation_v4 = (
    add_v4_derived_features(
        X_validation_v3
    )[V4_NUMERIC_FEATURES]
    .astype("float32")
)

X_calibration_v4 = (
    add_v4_derived_features(
        X_calibration_v3
    )[V4_NUMERIC_FEATURES]
    .astype("float32")
)

X_test_v4 = (
    add_v4_derived_features(
        X_test_v3
    )[V4_NUMERIC_FEATURES]
    .astype("float32")
)

y_train_v4 = y_train_v3.copy()
y_validation_v4 = y_validation_v3.copy()
y_calibration_v4 = y_calibration_v3.copy()
y_test_v4 = y_test_v3.copy()

print(
    "Fitur dasar V3:",
    len(V3_NUMERIC_FEATURES),
)

print(
    "Fitur konstan dibuang:",
    len(constant_features_v4),
)

print(
    "Fitur tambahan V4:",
    len(V4_DERIVED_FEATURES),
)

print(
    "Total fitur V4:",
    len(V4_NUMERIC_FEATURES),
)

Fitur dasar V3: 41
Fitur konstan dibuang: 0
Fitur tambahan V4: 15
Total fitur V4: 56


In [54]:
# ============================================================
# 08.2 CANDIDATE V4 MATRIX VALIDATION
# ============================================================

matrix_summary_v4 = []

for (
    split_name,
    matrix,
    target,
) in [
    (
        "train",
        X_train_v4,
        y_train_v4,
    ),
    (
        "validation",
        X_validation_v4,
        y_validation_v4,
    ),
    (
        "calibration",
        X_calibration_v4,
        y_calibration_v4,
    ),
    (
        "test",
        X_test_v4,
        y_test_v4,
    ),
]:
    missing_values = int(
        matrix
        .isna()
        .sum()
        .sum()
    )

    infinite_values = int(
        np.isinf(
            matrix.to_numpy()
        ).sum()
    )

    matrix_summary_v4.append({
        "split": split_name,
        "rows": matrix.shape[0],
        "features": matrix.shape[1],
        "fraud_rate": float(
            target.mean()
        ),
        "missing_values": (
            missing_values
        ),
        "infinite_values": (
            infinite_values
        ),
    })

matrix_summary_v4 = pd.DataFrame(
    matrix_summary_v4
)

display(
    matrix_summary_v4
)

assert (
    list(X_train_v4.columns)
    == V4_NUMERIC_FEATURES
)

assert (
    list(X_validation_v4.columns)
    == V4_NUMERIC_FEATURES
)

assert (
    list(X_calibration_v4.columns)
    == V4_NUMERIC_FEATURES
)

assert (
    list(X_test_v4.columns)
    == V4_NUMERIC_FEATURES
)

assert (
    matrix_summary_v4[
        "missing_values"
    ] == 0
).all()

assert (
    matrix_summary_v4[
        "infinite_values"
    ] == 0
).all()

print(
    "Matriks Candidate V4 siap untuk benchmark."
)

,split,rows,features,fraud_rate,missing_values,infinite_values
0,train,391716,56,0.312109,0,0
1,validation,58895,56,0.309160,0,0
2,calibration,55066,56,0.307177,0,0
3,test,63651,56,0.284944,0,0


Matriks Candidate V4 siap untuk benchmark.


In [55]:
# ============================================================
# 08.3 CANDIDATE V4 CONTROLLED LIGHTGBM BENCHMARK
# ============================================================

V4_COMMON_PARAMS = {
    "objective": "binary",
    "n_estimators": 2200,
    "learning_rate": 0.03,
    "max_bin": 255,
    "class_weight": None,
    "random_state": SEED,
    "n_jobs": -1,
    "force_row_wise": True,
    "verbosity": -1,
    "subsample_freq": 1,
}

V4_CANDIDATE_CONFIGS = {
    "conservative": {
        "num_leaves": 15,
        "max_depth": 5,
        "min_child_samples": 100,
        "min_split_gain": 0.0,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
        "reg_alpha": 0.50,
        "reg_lambda": 3.00,
    },
    "balanced": {
        "num_leaves": 31,
        "max_depth": 8,
        "min_child_samples": 50,
        "min_split_gain": 0.0,
        "subsample": 0.90,
        "colsample_bytree": 1.00,
        "reg_alpha": 0.10,
        "reg_lambda": 1.00,
    },
    "flexible": {
        "num_leaves": 63,
        "max_depth": 10,
        "min_child_samples": 30,
        "min_split_gain": 0.0,
        "subsample": 0.90,
        "colsample_bytree": 1.00,
        "reg_alpha": 0.00,
        "reg_lambda": 0.50,
    },
}

benchmark_models_v4 = {}
benchmark_rows_v4 = []

for (
    config_name,
    config_parameters,
) in V4_CANDIDATE_CONFIGS.items():

    print(
        "\nTraining configuration:",
        config_name,
    )

    model_parameters = {
        **V4_COMMON_PARAMS,
        **config_parameters,
    }

    candidate_model = LGBMClassifier(
        **model_parameters
    )

    candidate_model.fit(
        X_train_v4,
        y_train_v4,
        eval_set=[
            (
                X_validation_v4,
                y_validation_v4,
            )
        ],
        eval_metric="average_precision",
        callbacks=[
            early_stopping(
                stopping_rounds=100,
                first_metric_only=True,
                verbose=True,
            ),
            log_evaluation(
                period=50,
            ),
        ],
    )

    best_iteration = int(
        candidate_model.best_iteration_
        or V4_COMMON_PARAMS[
            "n_estimators"
        ]
    )

    validation_probabilities = (
        candidate_model.predict_proba(
            X_validation_v4,
            num_iteration=best_iteration,
        )[:, 1]
    )

    validation_pr_auc = float(
        average_precision_score(
            y_validation_v4,
            validation_probabilities,
        )
    )

    validation_operating_points = (
        build_operating_point_table(
            y_true=y_validation_v4,
            probabilities=(
                validation_probabilities
            ),
        )
    )

    recall_guardrail = (
        validation_operating_points[
            validation_operating_points[
                "recall"
            ] >= TARGET_RECALL
        ]
        .copy()
    )

    if recall_guardrail.empty:
        selected_validation_point = (
            validation_operating_points
            .sort_values(
                [
                    "f1",
                    "precision",
                ],
                ascending=[
                    False,
                    False,
                ],
            )
            .iloc[0]
        )
    else:
        selected_validation_point = (
            recall_guardrail
            .sort_values(
                [
                    "false_positive_rate",
                    "precision",
                    "f1",
                    "threshold",
                ],
                ascending=[
                    True,
                    False,
                    False,
                    False,
                ],
            )
            .iloc[0]
        )

    split_importance = (
        candidate_model
        .booster_
        .feature_importance(
            importance_type="split"
        )
    )

    used_feature_count = int(
        (
            split_importance > 0
        ).sum()
    )

    benchmark_models_v4[
        config_name
    ] = candidate_model

    benchmark_rows_v4.append({
        "config": config_name,
        "best_iteration": (
            best_iteration
        ),
        "validation_pr_auc": (
            validation_pr_auc
        ),
        "validation_pr_auc_lift": (
            validation_pr_auc
            / max(
                float(
                    y_validation_v4.mean()
                ),
                1e-12,
            )
        ),
        "precision_at_recall_70": float(
            selected_validation_point[
                "precision"
            ]
        ),
        "recall": float(
            selected_validation_point[
                "recall"
            ]
        ),
        "f1": float(
            selected_validation_point[
                "f1"
            ]
        ),
        "fpr_at_recall_70": float(
            selected_validation_point[
                "false_positive_rate"
            ]
        ),
        "threshold_at_recall_70": float(
            selected_validation_point[
                "threshold"
            ]
        ),
        "used_feature_count": (
            used_feature_count
        ),
    })

benchmark_results_v4 = (
    pd.DataFrame(
        benchmark_rows_v4
    )
    .sort_values(
        [
            "validation_pr_auc",
            "fpr_at_recall_70",
            "precision_at_recall_70",
        ],
        ascending=[
            False,
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)

best_config_name_v4 = str(
    benchmark_results_v4
    .iloc[0]["config"]
)

lightgbm_v4_selection_model = (
    benchmark_models_v4[
        best_config_name_v4
    ]
)

display(
    benchmark_results_v4
)

print(
    "Selected configuration:",
    best_config_name_v4,
)


Training configuration: conservative
Training until validation scores don't improve for 100 rounds
[50]	valid_0's average_precision: 0.341071	valid_0's binary_logloss: 0.607812
[100]	valid_0's average_precision: 0.356177	valid_0's binary_logloss: 0.606605
[150]	valid_0's average_precision: 0.3562	valid_0's binary_logloss: 0.606426
[200]	valid_0's average_precision: 0.356073	valid_0's binary_logloss: 0.606432
Early stopping, best iteration is:
[116]	valid_0's average_precision: 0.357413	valid_0's binary_logloss: 0.606496
Evaluated only: average_precision

Training configuration: balanced
Training until validation scores don't improve for 100 rounds
[50]	valid_0's average_precision: 0.355392	valid_0's binary_logloss: 0.607709
[100]	valid_0's average_precision: 0.358004	valid_0's binary_logloss: 0.606516
[150]	valid_0's average_precision: 0.356757	valid_0's binary_logloss: 0.606442
[200]	valid_0's average_precision: 0.356921	valid_0's binary_logloss: 0.606476
Early stopping, best iterati

,config,best_iteration,validation_pr_auc,validation_pr_auc_lift,precision_at_recall_70,recall,f1,fpr_at_recall_70,threshold_at_recall_70,used_feature_count
0,flexible,58,0.359096,1.161521,0.312498,0.700461,0.432185,0.689631,0.298874,43
1,balanced,110,0.358149,1.158458,0.311966,0.700022,0.431592,0.690909,0.298688,41
2,conservative,116,0.357413,1.156075,0.312360,0.714082,0.434610,0.703493,0.300472,39


Selected configuration: flexible


In [56]:
# ============================================================
# 08.4 CANDIDATE V4 REFIT TRAIN AND VALIDATION
# ============================================================

selected_best_iteration_v4 = int(
    lightgbm_v4_selection_model
    .best_iteration_
    or V4_COMMON_PARAMS[
        "n_estimators"
    ]
)

selected_parameters_v4 = {
    **V4_COMMON_PARAMS,
    **V4_CANDIDATE_CONFIGS[
        best_config_name_v4
    ],
}

selected_parameters_v4[
    "n_estimators"
] = selected_best_iteration_v4

X_refit_v4 = pd.concat(
    [
        X_train_v4,
        X_validation_v4,
    ],
    axis=0,
    ignore_index=True,
)

y_refit_v4 = pd.concat(
    [
        y_train_v4,
        y_validation_v4,
    ],
    axis=0,
    ignore_index=True,
)

lightgbm_v4_model = LGBMClassifier(
    **selected_parameters_v4
)

lightgbm_v4_model.fit(
    X_refit_v4,
    y_refit_v4,
)

selection_validation_probabilities_v4 = (
    lightgbm_v4_selection_model
    .predict_proba(
        X_validation_v4,
        num_iteration=(
            selected_best_iteration_v4
        ),
    )[:, 1]
)

selection_validation_pr_auc_v4 = float(
    average_precision_score(
        y_validation_v4,
        selection_validation_probabilities_v4,
    )
)

print(
    "Selected configuration:",
    best_config_name_v4,
)

print(
    "Selected best iteration:",
    selected_best_iteration_v4,
)

print(
    "Refit rows:",
    len(X_refit_v4),
)

print(
    "Selection validation PR-AUC:",
    selection_validation_pr_auc_v4,
)

Selected configuration: flexible
Selected best iteration: 58
Refit rows: 450611
Selection validation PR-AUC: 0.35909622841197264


In [57]:
# ============================================================
# 09.0 CANDIDATE V4 CALIBRATION THRESHOLD SELECTION
# ============================================================

calibration_probabilities_v4 = (
    lightgbm_v4_model
    .predict_proba(
        X_calibration_v4
    )[:, 1]
)

(
    calibration_summary_v4,
    calibration_operating_points_v4,
) = select_threshold_v4(
    model_name=(
        "LightGBM Candidate V4 Calibration"
    ),
    y_true=y_calibration_v4,
    probabilities=(
        calibration_probabilities_v4
    ),
)

selected_threshold_v4 = float(
    calibration_summary_v4
    .iloc[0]["threshold"]
)

display(
    calibration_summary_v4
)

print(
    "Selected threshold V4:",
    selected_threshold_v4,
)

,model,status,selection_strategy,pr_auc,threshold,precision,recall,f1,false_positive_rate,true_positive,false_positive,false_negative,true_negative
0,LightGBM Candidate V4 Calibration,TARGET NOT ACHIEVED,MINIMUM_FPR_AT_RECALL_GUARDRAIL,0.355613,0.303305,0.311773,0.700148,0.431431,0.685251,11843,26143,5072,12008


Selected threshold V4: 0.30330462799301994


In [58]:
# ============================================================
# 10.0 CANDIDATE V4 FINAL TEST EVALUATION
# ============================================================

test_probabilities_v4 = (
    lightgbm_v4_model
    .predict_proba(
        X_test_v4
    )[:, 1]
)

test_summary_v4 = (
    evaluate_fixed_threshold_v4(
        model_name=(
            "LightGBM Candidate V4 Final Test"
        ),
        y_true=y_test_v4,
        probabilities=(
            test_probabilities_v4
        ),
        threshold=(
            selected_threshold_v4
        ),
    )
)

test_predictions_v4 = (
    test_probabilities_v4
    >= selected_threshold_v4
).astype(np.int8)

tn_v4, fp_v4, fn_v4, tp_v4 = (
    confusion_matrix(
        y_test_v4,
        test_predictions_v4,
        labels=[0, 1],
    )
    .ravel()
)

test_confusion_matrix_v4 = pd.DataFrame(
    [
        {
            "actual_class": "NORMAL",
            "predicted_normal": int(
                tn_v4
            ),
            "predicted_attention": int(
                fp_v4
            ),
        },
        {
            "actual_class": (
                "SYNTHETIC_FRAUD"
            ),
            "predicted_normal": int(
                fn_v4
            ),
            "predicted_attention": int(
                tp_v4
            ),
        },
    ]
)

display(
    test_summary_v4
)

display(
    test_confusion_matrix_v4
)

print(
    classification_report(
        y_test_v4,
        test_predictions_v4,
        labels=[0, 1],
        target_names=[
            "normal",
            "synthetic_fraud",
        ],
        digits=6,
        zero_division=0,
    )
)

v4_test_row = (
    test_summary_v4.iloc[0]
)

comparison_v3_v4 = pd.DataFrame([
    {
        "model": "Candidate V3",
        **V3_BASELINE_TEST_METRICS,
    },
    {
        "model": "Candidate V4",
        "pr_auc": float(
            v4_test_row["pr_auc"]
        ),
        "precision": float(
            v4_test_row["precision"]
        ),
        "recall": float(
            v4_test_row["recall"]
        ),
        "f1": float(
            v4_test_row["f1"]
        ),
        "false_positive_rate": float(
            v4_test_row[
                "false_positive_rate"
            ]
        ),
        "alert_rate": float(
            v4_test_row["alert_rate"]
        ),
    },
])

display(
    comparison_v3_v4
)

,model,status,rows,fraud_rate,pr_auc,threshold,precision,precision_lift,recall,f1,false_positive_rate,alert_rate,true_positive,false_positive,false_negative,true_negative
0,LightGBM Candidate V4 Final Test,TARGET NOT ACHIEVED,63651,0.284944,0.351893,0.303305,0.311068,1.091679,0.800077,0.447967,0.706112,0.732887,14511,32138,3626,13376


,actual_class,predicted_normal,predicted_attention
0,NORMAL,13376,32138
1,SYNTHETIC_FRAUD,3626,14511


                 precision    recall  f1-score   support

         normal   0.786731  0.293888  0.427922     45514
synthetic_fraud   0.311068  0.800077  0.447967     18137

       accuracy                       0.438124     63651
      macro avg   0.548899  0.546982  0.437945     63651
   weighted avg   0.651193  0.438124  0.433634     63651



,model,pr_auc,precision,recall,f1,false_positive_rate,alert_rate
0,Candidate V3,0.348577,0.307330,0.998952,0.470048,0.897196,0.926191
1,Candidate V4,0.351893,0.311068,0.800077,0.447967,0.706112,0.732887


In [59]:
# ============================================================
# 10.1 CANDIDATE V4 PRECISION AND RECALL AT TOP K
# ============================================================

def calculate_top_k_metrics(
    y_true,
    probabilities,
    fractions,
):
    y_true = np.asarray(
        y_true,
        dtype=np.int8,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    sorted_indices = np.argsort(
        -probabilities,
        kind="mergesort",
    )

    total_positive = max(
        int(y_true.sum()),
        1,
    )

    rows = []

    for fraction in fractions:
        k = max(
            int(
                np.ceil(
                    len(y_true)
                    * fraction
                )
            ),
            1,
        )

        selected_indices = (
            sorted_indices[:k]
        )

        selected_positive = int(
            y_true[
                selected_indices
            ].sum()
        )

        rows.append({
            "top_fraction": fraction,
            "top_percent": (
                fraction * 100
            ),
            "transactions": k,
            "true_positive": (
                selected_positive
            ),
            "precision_at_k": (
                selected_positive
                / k
            ),
            "recall_at_k": (
                selected_positive
                / total_positive
            ),
        })

    return pd.DataFrame(rows)


top_k_metrics_v4 = (
    calculate_top_k_metrics(
        y_true=y_test_v4,
        probabilities=(
            test_probabilities_v4
        ),
        fractions=[
            0.01,
            0.05,
            0.10,
            0.20,
        ],
    )
)

display(
    top_k_metrics_v4
)

,top_fraction,top_percent,transactions,true_positive,precision_at_k,recall_at_k
0,0.01,1.0,637,566,0.888540,0.031207
1,0.05,5.0,3183,1315,0.413132,0.072504
2,0.10,10.0,6366,2321,0.364593,0.127970
3,0.20,20.0,12731,4218,0.331317,0.232563


In [60]:
# ============================================================
# 10.2 CANDIDATE V4 CALIBRATION-LOCKED TOP-FRACTION
# OPERATING POINTS
# ============================================================

GLOBAL_TOP_FRACTIONS_V4 = [
    0.01,
    0.02,
    0.05,
    0.10,
]


def calculate_fixed_threshold_metrics_v4(
    y_true,
    probabilities,
    threshold,
):
    y_true = np.asarray(
        y_true,
        dtype=np.int8,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    predictions = (
        probabilities >= threshold
    ).astype(np.int8)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    ).ravel()

    precision = (
        tp / max(tp + fp, 1)
    )

    recall = (
        tp / max(tp + fn, 1)
    )

    f1 = (
        2 * precision * recall
        / max(
            precision + recall,
            1e-12,
        )
    )

    false_positive_rate = (
        fp / max(fp + tn, 1)
    )

    return {
        "rows": len(y_true),
        "fraud_rate": float(
            y_true.mean()
        ),
        "alert_rate": float(
            predictions.mean()
        ),
        "precision": float(
            precision
        ),
        "recall": float(
            recall
        ),
        "f1": float(
            f1
        ),
        "false_positive_rate": float(
            false_positive_rate
        ),
        "true_positive": int(tp),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_negative": int(tn),
    }


top_fraction_rows_v4 = []

for configured_fraction in (
    GLOBAL_TOP_FRACTIONS_V4
):
    threshold = float(
        np.quantile(
            calibration_probabilities_v4,
            1.0 - configured_fraction,
        )
    )

    calibration_metrics = (
        calculate_fixed_threshold_metrics_v4(
            y_true=y_calibration_v4,
            probabilities=(
                calibration_probabilities_v4
            ),
            threshold=threshold,
        )
    )

    test_metrics = (
        calculate_fixed_threshold_metrics_v4(
            y_true=y_test_v4,
            probabilities=(
                test_probabilities_v4
            ),
            threshold=threshold,
        )
    )

    top_fraction_rows_v4.append({
        "configured_top_fraction": (
            configured_fraction
        ),
        "configured_top_percent": (
            configured_fraction * 100
        ),
        "threshold": threshold,
        "split": "calibration",
        **calibration_metrics,
    })

    top_fraction_rows_v4.append({
        "configured_top_fraction": (
            configured_fraction
        ),
        "configured_top_percent": (
            configured_fraction * 100
        ),
        "threshold": threshold,
        "split": "test",
        **test_metrics,
    })

top_fraction_operating_points_v4 = (
    pd.DataFrame(
        top_fraction_rows_v4
    )
    .sort_values(
        [
            "configured_top_fraction",
            "split",
        ]
    )
    .reset_index(drop=True)
)

display(
    top_fraction_operating_points_v4
)

,configured_top_fraction,configured_top_percent,threshold,split,rows,fraud_rate,alert_rate,precision,recall,f1,false_positive_rate,true_positive,false_positive,false_negative,true_negative
0,0.01,1.0,0.362449,calibration,55066,0.307177,0.010024,0.963768,0.031451,0.060915,0.000524,532,20,16383,38131
1,0.01,1.0,0.362449,test,63651,0.284944,0.008798,0.971429,0.029994,0.058191,0.000352,544,16,17593,45498
2,0.02,2.0,0.341170,calibration,55066,0.307177,0.020031,0.629193,0.041029,0.077034,0.010721,694,409,16221,37742
3,0.02,2.0,0.341170,test,63651,0.284944,0.021744,0.565029,0.043116,0.080119,0.013227,782,602,17355,44912
4,0.05,5.0,0.322711,calibration,55066,0.307177,0.050013,0.437545,0.071239,0.122528,0.040602,1205,1549,15710,36602
5,0.05,5.0,0.322711,test,63651,0.284944,0.114091,0.354172,0.141810,0.202528,0.103045,2572,4690,15565,40824
6,0.10,10.0,0.312198,calibration,55066,0.307177,0.100007,0.373525,0.121608,0.183481,0.090430,2057,3450,14858,34701
7,0.10,10.0,0.312198,test,63651,0.284944,0.177766,0.335484,0.209296,0.257775,0.165202,3796,7519,14341,37995


In [61]:
# ============================================================
# 10.3 CANDIDATE V4 GLOBAL SCORE PERCENTILE
# ============================================================

calibration_score_reference_v4 = np.sort(
    np.asarray(
        calibration_probabilities_v4,
        dtype=np.float64,
    )
)


def convert_to_calibration_percentile_v4(
    probabilities,
    reference_scores,
):
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    reference_scores = np.asarray(
        reference_scores,
        dtype=np.float64,
    )

    percentile_positions = np.searchsorted(
        reference_scores,
        probabilities,
        side="right",
    )

    return (
        percentile_positions
        / max(
            len(reference_scores),
            1,
        )
    )


test_global_percentile_v4 = (
    convert_to_calibration_percentile_v4(
        probabilities=(
            test_probabilities_v4
        ),
        reference_scores=(
            calibration_score_reference_v4
        ),
    )
)

test_global_signal_level_v4 = np.select(
    [
        test_global_percentile_v4
        >= 0.99,
        test_global_percentile_v4
        >= 0.95,
    ],
    [
        "STRONG_GLOBAL_SIGNAL",
        "SUPPORTING_GLOBAL_SIGNAL",
    ],
    default="LOW_GLOBAL_SIGNAL",
)

global_percentile_diagnostic_v4 = (
    pd.DataFrame({
        "target": (
            np.asarray(
                y_test_v4,
                dtype=np.int8,
            )
        ),
        "global_probability": (
            test_probabilities_v4
        ),
        "global_percentile": (
            test_global_percentile_v4
        ),
        "global_signal_level": (
            test_global_signal_level_v4
        ),
    })
)

global_signal_summary_v4 = (
    global_percentile_diagnostic_v4
    .groupby(
        "global_signal_level",
        as_index=False,
    )
    .agg(
        transaction_count=(
            "target",
            "size",
        ),
        synthetic_fraud_count=(
            "target",
            "sum",
        ),
        synthetic_fraud_rate=(
            "target",
            "mean",
        ),
        mean_probability=(
            "global_probability",
            "mean",
        ),
        minimum_percentile=(
            "global_percentile",
            "min",
        ),
        maximum_percentile=(
            "global_percentile",
            "max",
        ),
    )
    .sort_values(
        "minimum_percentile",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    global_signal_summary_v4
)

,global_signal_level,transaction_count,synthetic_fraud_count,synthetic_fraud_rate,mean_probability,minimum_percentile,maximum_percentile
0,STRONG_GLOBAL_SIGNAL,560,544,0.971429,0.855005,0.990121,1.000000
1,SUPPORTING_GLOBAL_SIGNAL,6702,2028,0.302596,0.336094,0.950005,0.989958
2,LOW_GLOBAL_SIGNAL,56389,15565,0.276029,0.285412,0.002579,0.949987


In [62]:
# ============================================================
# 11.0 CANDIDATE V4 FEATURE IMPORTANCE
# ============================================================

feature_importance_v4 = pd.DataFrame({
    "feature": (
        lightgbm_v4_model
        .booster_
        .feature_name()
    ),
    "importance_gain": (
        lightgbm_v4_model
        .booster_
        .feature_importance(
            importance_type="gain"
        )
    ),
    "importance_split": (
        lightgbm_v4_model
        .booster_
        .feature_importance(
            importance_type="split"
        )
    ),
})

total_gain_v4 = float(
    feature_importance_v4[
        "importance_gain"
    ].sum()
)

feature_importance_v4[
    "gain_percentage"
] = np.where(
    total_gain_v4 > 0,
    (
        feature_importance_v4[
            "importance_gain"
        ]
        / total_gain_v4
        * 100
    ),
    0,
)

feature_importance_v4 = (
    feature_importance_v4
    .sort_values(
        "importance_gain",
        ascending=False,
    )
    .reset_index(drop=True)
)

used_feature_count_v4 = int(
    (
        feature_importance_v4[
            "importance_split"
        ] > 0
    ).sum()
)

zero_importance_count_v4 = int(
    (
        feature_importance_v4[
            "importance_split"
        ] == 0
    ).sum()
)

top_1_gain_share_v4 = float(
    feature_importance_v4[
        "gain_percentage"
    ].head(1).sum()
)

top_5_gain_share_v4 = float(
    feature_importance_v4[
        "gain_percentage"
    ].head(5).sum()
)

top_10_gain_share_v4 = float(
    feature_importance_v4[
        "gain_percentage"
    ].head(10).sum()
)

display(
    feature_importance_v4.head(25)
)

print(
    "Fitur digunakan:",
    used_feature_count_v4,
)

print(
    "Fitur importance nol:",
    zero_importance_count_v4,
)

print(
    "Top 1 gain share:",
    top_1_gain_share_v4,
)

print(
    "Top 5 gain share:",
    top_5_gain_share_v4,
)

print(
    "Top 10 gain share:",
    top_10_gain_share_v4,
)

,feature,importance_gain,importance_split,gain_percentage
0,initiator_amount_max_prior,210351.206396,146,37.664367
1,amount,125385.565759,245,22.450872
2,new_bal_initiator,99755.085209,146,17.861615
3,amount_vs_initiator_mean,36336.739849,124,6.506263
4,recipient_tx_count_prior,13686.367702,379,2.450608
5,old_bal_initiator,10620.697805,166,1.901686
6,remaining_balance_ratio,10533.775440,131,1.886122
7,amount_log_ratio_to_prior_max,8894.743426,120,1.592645
8,initiator_balance_change,7880.100250,124,1.410969
9,hour_of_day,4272.136418,173,0.764946


Fitur digunakan: 40
Fitur importance nol: 16
Top 1 gain share: 37.66436734545747
Top 5 gain share: 86.93372528548578
Top 10 gain share: 94.4900928877826


In [63]:
# ============================================================
# 11.1 CANDIDATE V4 TEMPORAL STABILITY DIAGNOSTIC
# ============================================================

def build_temporal_diagnostic_v4(
    split_name,
    source_dataframe,
    targets,
    probabilities,
    threshold,
):
    diagnostic_dataframe = pd.DataFrame({
        "step": (
            source_dataframe[
                "step"
            ]
            .to_numpy()
        ),
        "target": np.asarray(
            targets,
            dtype=np.int8,
        ),
        "probability": np.asarray(
            probabilities,
            dtype=np.float64,
        ),
    })

    diagnostic_dataframe[
        "prediction"
    ] = (
        diagnostic_dataframe[
            "probability"
        ] >= threshold
    ).astype(np.int8)

    rows = []

    for step_value, group in (
        diagnostic_dataframe
        .groupby(
            "step",
            sort=True,
        )
    ):
        y_step = (
            group["target"]
            .to_numpy()
        )

        probability_step = (
            group["probability"]
            .to_numpy()
        )

        prediction_step = (
            group["prediction"]
            .to_numpy()
        )

        if np.unique(y_step).size >= 2:
            step_pr_auc = float(
                average_precision_score(
                    y_step,
                    probability_step,
                )
            )
        else:
            step_pr_auc = np.nan

        tn, fp, fn, tp = confusion_matrix(
            y_step,
            prediction_step,
            labels=[0, 1],
        ).ravel()

        rows.append({
            "split": split_name,
            "step": int(step_value),
            "rows": int(
                len(group)
            ),
            "fraud_rate": float(
                y_step.mean()
            ),
            "mean_probability": float(
                probability_step.mean()
            ),
            "pr_auc": step_pr_auc,
            "alert_rate": float(
                prediction_step.mean()
            ),
            "precision": float(
                tp / max(tp + fp, 1)
            ),
            "recall": float(
                tp / max(tp + fn, 1)
            ),
            "false_positive_rate": float(
                fp / max(fp + tn, 1)
            ),
        })

    return pd.DataFrame(rows)


calibration_temporal_diagnostic_v4 = (
    build_temporal_diagnostic_v4(
        split_name="calibration",
        source_dataframe=(
            calibration_v3_df
        ),
        targets=y_calibration_v4,
        probabilities=(
            calibration_probabilities_v4
        ),
        threshold=(
            selected_threshold_v4
        ),
    )
)

test_temporal_diagnostic_v4 = (
    build_temporal_diagnostic_v4(
        split_name="test",
        source_dataframe=test_v3_df,
        targets=y_test_v4,
        probabilities=(
            test_probabilities_v4
        ),
        threshold=(
            selected_threshold_v4
        ),
    )
)

temporal_diagnostic_v4 = pd.concat(
    [
        calibration_temporal_diagnostic_v4,
        test_temporal_diagnostic_v4,
    ],
    axis=0,
    ignore_index=True,
)

display(
    temporal_diagnostic_v4
)

,split,step,rows,fraud_rate,mean_probability,pr_auc,alert_rate,precision,recall,false_positive_rate
0,calibration,115,3891,0.302493,0.309745,0.356745,0.640709,0.313277,0.663551,0.630803
1,calibration,116,3881,0.301726,0.311499,0.350718,0.666065,0.308317,0.680615,0.659779
2,calibration,117,3887,0.316954,0.310810,0.365112,0.632622,0.318422,0.635552,0.631262
3,calibration,118,3857,0.305678,0.310115,0.337656,0.637542,0.300529,0.626802,0.642270
4,calibration,119,4108,0.289435,0.303026,0.357473,0.645813,0.307199,0.685450,0.629668
5,calibration,120,4017,0.290266,0.306636,0.345935,0.608414,0.296645,0.621784,0.602946
6,calibration,121,3988,0.306670,0.311465,0.367209,0.657974,0.315930,0.677841,0.649186
7,calibration,122,3948,0.307751,0.309280,0.345109,0.672239,0.307837,0.672428,0.672155
8,calibration,123,3969,0.317964,0.311612,0.364825,0.684303,0.325479,0.700475,0.676764
9,calibration,124,3928,0.312118,0.312275,0.346646,0.738289,0.309310,0.731648,0.741303


In [67]:
# ============================================================
# 11.2 CANDIDATE V4 ALERT RATE CONSISTENCY CHECK
# ============================================================

summary_alert_rate_v4 = float(
    test_summary_v4
    .iloc[0]["alert_rate"]
)

prediction_alert_rate_v4 = float(
    np.asarray(
        test_predictions_v4,
        dtype=np.int8,
    ).mean()
)

weighted_temporal_alert_rate_v4 = float(
    np.average(
        test_temporal_diagnostic_v4[
            "alert_rate"
        ],
        weights=(
            test_temporal_diagnostic_v4[
                "rows"
            ]
        ),
    )
)

summary_vs_prediction_difference_v4 = abs(
    summary_alert_rate_v4
    - prediction_alert_rate_v4
)

summary_vs_temporal_difference_v4 = abs(
    summary_alert_rate_v4
    - weighted_temporal_alert_rate_v4
)

prediction_vs_temporal_difference_v4 = abs(
    prediction_alert_rate_v4
    - weighted_temporal_alert_rate_v4
)

alert_rate_consistency_v4 = pd.DataFrame([
    {
        "source": (
            "test_summary_v4"
        ),
        "alert_rate": (
            summary_alert_rate_v4
        ),
    },
    {
        "source": (
            "test_predictions_v4.mean"
        ),
        "alert_rate": (
            prediction_alert_rate_v4
        ),
    },
    {
        "source": (
            "weighted_test_temporal_diagnostic"
        ),
        "alert_rate": (
            weighted_temporal_alert_rate_v4
        ),
    },
])

display(
    alert_rate_consistency_v4
)

print(
    "Difference summary vs prediction:",
    summary_vs_prediction_difference_v4,
)

print(
    "Difference summary vs temporal:",
    summary_vs_temporal_difference_v4,
)

print(
    "Difference prediction vs temporal:",
    prediction_vs_temporal_difference_v4,
)

is_summary_prediction_consistent_v4 = (
    np.isclose(
        summary_alert_rate_v4,
        prediction_alert_rate_v4,
        atol=1e-12,
        rtol=0.0,
    )
)

is_summary_temporal_consistent_v4 = (
    np.isclose(
        summary_alert_rate_v4,
        weighted_temporal_alert_rate_v4,
        atol=1e-12,
        rtol=0.0,
    )
)

is_prediction_temporal_consistent_v4 = (
    np.isclose(
        prediction_alert_rate_v4,
        weighted_temporal_alert_rate_v4,
        atol=1e-12,
        rtol=0.0,
    )
)

all_alert_rates_consistent_v4 = (
    is_summary_prediction_consistent_v4
    and is_summary_temporal_consistent_v4
    and is_prediction_temporal_consistent_v4
)

print(
    "Summary dan prediction konsisten:",
    is_summary_prediction_consistent_v4,
)

print(
    "Summary dan temporal konsisten:",
    is_summary_temporal_consistent_v4,
)

print(
    "Prediction dan temporal konsisten:",
    is_prediction_temporal_consistent_v4,
)

print(
    "Semua alert rate konsisten:",
    all_alert_rates_consistent_v4,
)

if not all_alert_rates_consistent_v4:
    print(
        "\nWARNING:"
        "\nVariabel berasal dari run atau model yang berbeda."
        "\nRestart session dan jalankan ulang cell 08.3 sampai 11.2 "
        "secara berurutan."
    )
else:
    print(
        "\nAlert rate konsisten. "
        "Artefak aman untuk diekspor dari run ini."
    )

,source,alert_rate
0,test_summary_v4,0.732887
1,test_predictions_v4.mean,0.732887
2,weighted_test_temporal_diagnostic,0.732887


Difference summary vs prediction: 0.0
Difference summary vs temporal: 0.0
Difference prediction vs temporal: 0.0
Summary dan prediction konsisten: True
Summary dan temporal konsisten: True
Prediction dan temporal konsisten: True
Semua alert rate konsisten: True

Alert rate konsisten. Artefak aman untuk diekspor dari run ini.


In [64]:
# ============================================================
# 12.0 EXPORT GLOBAL RANKER V4 ARTIFACTS
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

GLOBAL_RANKER_NAME = (
    "global_ranker_v4_momtsim_v1"
)

GLOBAL_RANKER_VERSION = "4.0.0"

GLOBAL_RANKER_DIR = (
    Path(MODEL_DIR)
    / GLOBAL_RANKER_NAME
)

GLOBAL_RANKER_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(
    "Export directory:",
    GLOBAL_RANKER_DIR,
)

# ============================================================
# FILE PATHS
# ============================================================

MODEL_PATH_V4 = (
    GLOBAL_RANKER_DIR
    / "global_ranker_v4.joblib"
)

FEATURE_CONTRACT_PATH_V4 = (
    GLOBAL_RANKER_DIR
    / "feature_contract_v4.json"
)

METADATA_PATH_V4 = (
    GLOBAL_RANKER_DIR
    / "model_metadata_v4.json"
)

CALIBRATION_REFERENCE_PATH_V4 = (
    GLOBAL_RANKER_DIR
    / "calibration_score_reference_v4.npy"
)

CALIBRATION_THRESHOLD_PATH_V4 = (
    GLOBAL_RANKER_DIR
    / "calibration_threshold_v4.json"
)

TOP_FRACTION_THRESHOLDS_PATH_V4 = (
    GLOBAL_RANKER_DIR
    / "top_fraction_thresholds_v4.json"
)

EVALUATION_METRICS_PATH_V4 = (
    GLOBAL_RANKER_DIR
    / "evaluation_metrics_v4.json"
)

FEATURE_IMPORTANCE_PATH_V4 = (
    GLOBAL_RANKER_DIR
    / "feature_importance_v4.csv"
)

BENCHMARK_RESULTS_PATH_V4 = (
    GLOBAL_RANKER_DIR
    / "benchmark_results_v4.csv"
)

FEATURE_DIAGNOSTIC_PATH_V4 = (
    GLOBAL_RANKER_DIR
    / "feature_diagnostic_v4.csv"
)

TOP_K_METRICS_PATH_V4 = (
    GLOBAL_RANKER_DIR
    / "top_k_metrics_v4.csv"
)

# ============================================================
# SAVE LIGHTGBM MODEL
# ============================================================

joblib.dump(
    lightgbm_v4_model,
    MODEL_PATH_V4,
)

# ============================================================
# SAVE FEATURE CONTRACT
# ============================================================

feature_contract_v4 = {
    "feature_contract_version": "4.0.0",
    "base_feature_version": "candidate_v3",
    "model_scope": "TRANSFER",
    "numeric_features": list(
        V4_NUMERIC_FEATURES
    ),
    "base_features": list(
        V4_BASE_FEATURES
    ),
    "derived_features": list(
        V4_DERIVED_FEATURES
    ),
    "removed_constant_features": list(
        constant_features_v4
    ),
    "feature_count": int(
        len(V4_NUMERIC_FEATURES)
    ),
    "required_audit_columns": [
        "step",
        "initiator",
        "recipient",
        "transaction_type",
    ],
    "target_column": "target",
    "unsupported_production_features": [
        "old_bal_recipient",
        "new_bal_recipient",
        "recipient_balance_change",
        "recipient_balance_error",
        "amount_to_recipient_balance_ratio",
        "recipient_old_balance_zero",
    ],
}

with open(
    FEATURE_CONTRACT_PATH_V4,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        feature_contract_v4,
        file,
        indent=2,
        ensure_ascii=False,
    )

# ============================================================
# SAVE CALIBRATION SCORE REFERENCE
# ============================================================

calibration_score_reference_v4 = (
    np.sort(
        np.asarray(
            calibration_probabilities_v4,
            dtype=np.float64,
        )
    )
)

np.save(
    CALIBRATION_REFERENCE_PATH_V4,
    calibration_score_reference_v4,
)

# ============================================================
# SAVE CALIBRATION THRESHOLD
# Threshold ini hasil eksperimen global classifier.
# Jangan langsung dipakai sebagai threshold produk.
# ============================================================

calibration_threshold_config_v4 = {
    "threshold_version": "4.0.0",
    "threshold": float(
        selected_threshold_v4
    ),
    "selection_split": "calibration",
    "selection_strategy": str(
        calibration_summary_v4.iloc[0][
            "selection_strategy"
        ]
    ),
    "recommended_product_usage": False,
    "usage_note": (
        "Threshold eksperimen global classifier. "
        "Global ranker V4 direkomendasikan sebagai "
        "ranking signal dan input score fusion."
    ),
}

with open(
    CALIBRATION_THRESHOLD_PATH_V4,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        calibration_threshold_config_v4,
        file,
        indent=2,
        ensure_ascii=False,
    )

# ============================================================
# SAVE CALIBRATION-LOCKED TOP-FRACTION THRESHOLDS
# ============================================================

top_fraction_thresholds_v4 = {}

for configured_fraction in (
    GLOBAL_TOP_FRACTIONS_V4
):
    threshold_value = float(
        np.quantile(
            calibration_probabilities_v4,
            1.0 - configured_fraction,
        )
    )

    threshold_key = (
        f"top_{int(configured_fraction * 100)}"
        "_percent"
    )

    top_fraction_thresholds_v4[
        threshold_key
    ] = {
        "fraction": float(
            configured_fraction
        ),
        "percent": float(
            configured_fraction * 100
        ),
        "threshold": threshold_value,
        "reference_split": "calibration",
    }

with open(
    TOP_FRACTION_THRESHOLDS_PATH_V4,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        top_fraction_thresholds_v4,
        file,
        indent=2,
        ensure_ascii=False,
    )

# ============================================================
# SAVE TEST EVALUATION
# ============================================================

test_metrics_v4 = (
    test_summary_v4
    .iloc[0]
    .to_dict()
)

evaluation_metrics_v4 = {
    key: (
        value.item()
        if isinstance(
            value,
            np.generic,
        )
        else value
    )
    for key, value in (
        test_metrics_v4.items()
    )
}

evaluation_metrics_v4[
    "top_k_metrics"
] = (
    top_k_metrics_v4
    .to_dict(
        orient="records"
    )
)

evaluation_metrics_v4[
    "baseline_candidate_v3"
] = V3_BASELINE_TEST_METRICS

with open(
    EVALUATION_METRICS_PATH_V4,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        evaluation_metrics_v4,
        file,
        indent=2,
        ensure_ascii=False,
    )

# ============================================================
# SAVE MODEL METADATA
# ============================================================

model_metadata_v4 = {
    "model_name": GLOBAL_RANKER_NAME,
    "model_version": (
        GLOBAL_RANKER_VERSION
    ),
    "experiment_version": (
        EXPERIMENT_VERSION
    ),
    "dataset_version": (
        DATASET_VERSION
    ),
    "model_type": "LightGBM",
    "model_role": "global_transaction_risk_ranker",
    "decision_scope": "TRANSFER",
    "label_column": "is_fraud",
    "label_meaning": (
        "Synthetic fraud generated by MoMTSim. "
        "Not a gambling label."
    ),
    "feature_contract_version": "4.0.0",
    "feature_count": int(
        len(V4_NUMERIC_FEATURES)
    ),
    "selected_configuration": (
        best_config_name_v4
    ),
    "best_iteration": int(
        selected_best_iteration_v4
    ),
    "calibration_threshold": float(
        selected_threshold_v4
    ),
    "recommended_usage": [
        "global probability",
        "calibration percentile",
        "top-fraction ranking signal",
        "input for Model 1 score fusion",
    ],
    "not_recommended_usage": [
        "standalone fraud decision",
        "standalone gambling detection",
        "standalone user alert",
    ],
    "created_at_utc": (
        datetime.now(
            timezone.utc
        )
        .isoformat()
    ),
}

with open(
    METADATA_PATH_V4,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        model_metadata_v4,
        file,
        indent=2,
        ensure_ascii=False,
    )

# ============================================================
# SAVE DIAGNOSTIC TABLES
# ============================================================

feature_importance_v4.to_csv(
    FEATURE_IMPORTANCE_PATH_V4,
    index=False,
)

benchmark_results_v4.to_csv(
    BENCHMARK_RESULTS_PATH_V4,
    index=False,
)

feature_diagnostic_v4.to_csv(
    FEATURE_DIAGNOSTIC_PATH_V4,
    index=False,
)

top_k_metrics_v4.to_csv(
    TOP_K_METRICS_PATH_V4,
    index=False,
)

print(
    "Global ranker V4 berhasil diekspor."
)

print(
    "Model:",
    MODEL_PATH_V4,
)

print(
    "Feature contract:",
    FEATURE_CONTRACT_PATH_V4,
)

print(
    "Metadata:",
    METADATA_PATH_V4,
)

print(
    "Calibration reference:",
    CALIBRATION_REFERENCE_PATH_V4,
)

print(
    "Top-fraction thresholds:",
    TOP_FRACTION_THRESHOLDS_PATH_V4,
)

Export directory: /content/drive/MyDrive/favera/model/transaction-anomaly/models/global_ranker_v4_momtsim_v1
Global ranker V4 berhasil diekspor.
Model: /content/drive/MyDrive/favera/model/transaction-anomaly/models/global_ranker_v4_momtsim_v1/global_ranker_v4.joblib
Feature contract: /content/drive/MyDrive/favera/model/transaction-anomaly/models/global_ranker_v4_momtsim_v1/feature_contract_v4.json
Metadata: /content/drive/MyDrive/favera/model/transaction-anomaly/models/global_ranker_v4_momtsim_v1/model_metadata_v4.json
Calibration reference: /content/drive/MyDrive/favera/model/transaction-anomaly/models/global_ranker_v4_momtsim_v1/calibration_score_reference_v4.npy
Top-fraction thresholds: /content/drive/MyDrive/favera/model/transaction-anomaly/models/global_ranker_v4_momtsim_v1/top_fraction_thresholds_v4.json


In [65]:
# ============================================================
# 12.1 VERIFY EXPORTED GLOBAL RANKER V4
# ============================================================

required_artifact_paths_v4 = [
    MODEL_PATH_V4,
    FEATURE_CONTRACT_PATH_V4,
    METADATA_PATH_V4,
    CALIBRATION_REFERENCE_PATH_V4,
    CALIBRATION_THRESHOLD_PATH_V4,
    TOP_FRACTION_THRESHOLDS_PATH_V4,
    EVALUATION_METRICS_PATH_V4,
    FEATURE_IMPORTANCE_PATH_V4,
    BENCHMARK_RESULTS_PATH_V4,
    FEATURE_DIAGNOSTIC_PATH_V4,
    TOP_K_METRICS_PATH_V4,
]

missing_artifacts_v4 = [
    str(path)
    for path in required_artifact_paths_v4
    if not path.exists()
]

if missing_artifacts_v4:
    raise FileNotFoundError(
        "Artefak V4 tidak ditemukan:\n"
        + "\n".join(
            missing_artifacts_v4
        )
    )

loaded_global_ranker_v4 = (
    joblib.load(
        MODEL_PATH_V4
    )
)

with open(
    FEATURE_CONTRACT_PATH_V4,
    "r",
    encoding="utf-8",
) as file:
    loaded_feature_contract_v4 = (
        json.load(file)
    )

with open(
    METADATA_PATH_V4,
    "r",
    encoding="utf-8",
) as file:
    loaded_metadata_v4 = (
        json.load(file)
    )

loaded_calibration_reference_v4 = (
    np.load(
        CALIBRATION_REFERENCE_PATH_V4
    )
)

if (
    loaded_feature_contract_v4[
        "numeric_features"
    ]
    != list(V4_NUMERIC_FEATURES)
):
    raise ValueError(
        "Feature contract hasil export "
        "tidak sama dengan fitur training."
    )

if (
    loaded_global_ranker_v4
    .n_features_in_
    != len(V4_NUMERIC_FEATURES)
):
    raise ValueError(
        "Jumlah fitur pada model export "
        "tidak sesuai feature contract."
    )

if (
    loaded_calibration_reference_v4.ndim
    != 1
):
    raise ValueError(
        "Calibration reference harus "
        "berupa array satu dimensi."
    )

verification_probabilities_v4 = (
    loaded_global_ranker_v4
    .predict_proba(
        X_test_v4.head(20)
    )[:, 1]
)

original_probabilities_sample_v4 = (
    test_probabilities_v4[:20]
)

maximum_probability_difference_v4 = float(
    np.max(
        np.abs(
            verification_probabilities_v4
            - original_probabilities_sample_v4
        )
    )
)

if (
    maximum_probability_difference_v4
    > 1e-10
):
    raise ValueError(
        "Prediksi model setelah reload "
        "berbeda dari model asli."
    )

artifact_verification_v4 = pd.DataFrame([
    {
        "artifact": (
            path.name
        ),
        "exists": (
            path.exists()
        ),
        "size_bytes": (
            path.stat().st_size
        ),
    }
    for path in required_artifact_paths_v4
])

display(
    artifact_verification_v4
)

print(
    "Model name:",
    loaded_metadata_v4[
        "model_name"
    ],
)

print(
    "Feature count:",
    loaded_feature_contract_v4[
        "feature_count"
    ],
)

print(
    "Calibration reference rows:",
    len(
        loaded_calibration_reference_v4
    ),
)

print(
    "Maximum reload prediction difference:",
    maximum_probability_difference_v4,
)

print(
    "Verifikasi artefak V4 berhasil."
)

,artifact,exists,size_bytes
0,global_ranker_v4.joblib,True,410278
1,feature_contract_v4.json,True,4051
2,model_metadata_v4.json,True,910
3,calibration_score_reference_v4.npy,True,440656
4,calibration_threshold_v4.json,True,343
5,top_fraction_thresholds_v4.json,True,570
6,evaluation_metrics_v4.json,True,1584
7,feature_importance_v4.csv,True,3190
8,benchmark_results_v4.csv,True,610
9,feature_diagnostic_v4.csv,True,8345


Model name: global_ranker_v4_momtsim_v1
Feature count: 56
Calibration reference rows: 55066
Maximum reload prediction difference: 0.0
Verifikasi artefak V4 berhasil.


In [66]:
# ============================================================
# 12.2 GLOBAL RANKER V4 INFERENCE INTERFACE
# ============================================================

def convert_score_to_percentile_v4(
    probability,
    calibration_reference,
):
    calibration_reference = np.asarray(
        calibration_reference,
        dtype=np.float64,
    )

    position = np.searchsorted(
        calibration_reference,
        float(probability),
        side="right",
    )

    percentile = (
        position
        / max(
            len(calibration_reference),
            1,
        )
    )

    return float(
        np.clip(
            percentile,
            0.0,
            1.0,
        )
    )


def map_global_signal_level_v4(
    percentile,
):
    if percentile >= 0.99:
        return "STRONG_GLOBAL_SIGNAL"

    if percentile >= 0.95:
        return "SUPPORTING_GLOBAL_SIGNAL"

    return "LOW_GLOBAL_SIGNAL"


def predict_global_ranker_v4(
    feature_row,
    model,
    feature_contract,
    calibration_reference,
):
    expected_features = (
        feature_contract[
            "numeric_features"
        ]
    )

    missing_features = [
        feature
        for feature in expected_features
        if feature not in feature_row.columns
    ]

    if missing_features:
        raise ValueError(
            "Fitur inference tidak lengkap: "
            + str(
                missing_features
            )
        )

    model_matrix = (
        feature_row[
            expected_features
        ]
        .astype("float32")
    )

    probability = float(
        model.predict_proba(
            model_matrix
        )[0, 1]
    )

    percentile = (
        convert_score_to_percentile_v4(
            probability=probability,
            calibration_reference=(
                calibration_reference
            ),
        )
    )

    signal_level = (
        map_global_signal_level_v4(
            percentile
        )
    )

    return {
        "global_probability": probability,
        "global_percentile": percentile,
        "global_signal_level": (
            signal_level
        ),
        "model_name": (
            GLOBAL_RANKER_NAME
        ),
        "model_version": (
            GLOBAL_RANKER_VERSION
        ),
    }


global_inference_example_v4 = (
    predict_global_ranker_v4(
        feature_row=(
            X_test_v4.iloc[
                [0]
            ]
        ),
        model=(
            loaded_global_ranker_v4
        ),
        feature_contract=(
            loaded_feature_contract_v4
        ),
        calibration_reference=(
            loaded_calibration_reference_v4
        ),
    )
)

print(
    json.dumps(
        global_inference_example_v4,
        indent=2,
        ensure_ascii=False,
    )
)

{
  "global_probability": 0.14975735046517433,
  "global_percentile": 0.004213126066901537,
  "global_signal_level": "LOW_GLOBAL_SIGNAL",
  "model_name": "global_ranker_v4_momtsim_v1",
  "model_version": "4.0.0"
}
